# AnimalCLEF2026 Final Texas Boundary Split-Only v20260430

This notebook is designed for the final one-submission decision.

## Base

It starts from the current best public-LB lineage:

```text
0.32583 = 0.31811 base + Salamander p80 cap
```

## Problem

The Texas no-oval branch improved the score because it preserved belly/near-belly dots, but visual inspection showed a flaw: the animal boundary can become a strong false fingerprint. The old hard oval was too rigid and discarded useful dots when the lizard size/alignment varied.

## Fix

Use the full aligned lizard foreground, but downweight the silhouette using the distance transform of the animal mask:

```text
W(x) = 0.08 + 0.92 * clip((D(x) - r0) / (r1 - r0), 0, 1)^0.72
H_final(x) = clip(H_raw(x) * W(x) - 0.20 * EdgeBand(x), 0, 1)
```

Then use the result only as a **split verifier**:

```text
start from current 0.32583
for Texas only:
    score pairs inside each existing cluster
    keep trusted dot-layout edges
    split weakly connected current clusters
    never merge different current clusters
```

This notebook cannot create the previous bad `max Texas cluster 195` collapse because it has no cross-cluster merge path.

## Required Kaggle Inputs

Attach:

1. `animal-clef-2026`
2. SAM export output containing:
   - `animalclef_sam3_views_cache/reports/view_manifest_sam3_all_species.csv`
3. Current-best LB `0.32583` output containing one of:
   - `submission_032583_salamander_p80_cluster_cap_v20260429.csv`
   - `submission_032368_gamble_salamander_p80_cap_v20260429.csv`
   - top-level `submission.csv` from the LB032583 notebook

Recommended current-best source:

```text
hanifnoerrofiq/lb-0-32583-salamander-p80-cap-v20260429
```

CPU is enough. No training and no SAM recomputation.
            

In [ ]:
from pathlib import Path

Path("species_fingerprint_final_swing_v20260426.py").write_text("from __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nimport math\nimport random\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageFile\n\n\nImageFile.LOAD_TRUNCATED_IMAGES = True\n\nVERSION = \"species_fingerprint_final_swing_v20260426\"\nSEED = 20260426\nBACKGROUND_RGB = np.array([238, 238, 232], dtype=np.float32)\n\nSPECIES_ORDER = [\n    \"LynxID2025\",\n    \"SalamanderID2025\",\n    \"SeaTurtleID2022\",\n    \"TexasHornedLizards\",\n]\n\nCURRENT_BEST_FILENAME = \"submission_hybrid_v3rescue_p06_salamander_turtle_v20260425.csv\"\n\n\n@dataclass(frozen=True)\nclass EdgeRule:\n    name: str\n    min_inliers: int\n    min_inlier_ratio: float\n    min_score: float\n    allow_lynx_opposite_side: bool = False\n\n\n@dataclass(frozen=True)\nclass SpeciesConfig:\n    species: str\n    roi_kind: str\n    target_size: tuple[int, int]\n    top_k: int\n    max_keypoints: int\n    ratio_test: float\n    ransac_reproj: float\n    split_large_at: int\n    preserve_clusters_up_to: int\n    max_cluster_pair_size: int\n    conservative: EdgeRule\n    balanced: EdgeRule\n    swing: EdgeRule\n\n\nSPECIES_CONFIGS: dict[str, SpeciesConfig] = {\n    \"LynxID2025\": SpeciesConfig(\n        species=\"LynxID2025\",\n        roi_kind=\"lynx_flank_spots\",\n        target_size=(448, 288),\n        top_k=34,\n        max_keypoints=900,\n        ratio_test=0.78,\n        ransac_reproj=5.0,\n        split_large_at=42,\n        preserve_clusters_up_to=18,\n        max_cluster_pair_size=90,\n        conservative=EdgeRule(\"conservative\", min_inliers=24, min_inlier_ratio=0.22, min_score=0.42),\n        balanced=EdgeRule(\"balanced\", min_inliers=15, min_inlier_ratio=0.16, min_score=0.30),\n        swing=EdgeRule(\n            \"swing\",\n            min_inliers=9,\n            min_inlier_ratio=0.11,\n            min_score=0.20,\n            allow_lynx_opposite_side=False,\n        ),\n    ),\n    \"SalamanderID2025\": SpeciesConfig(\n        species=\"SalamanderID2025\",\n        roi_kind=\"salamander_straight_strip\",\n        target_size=(544, 176),\n        top_k=30,\n        max_keypoints=850,\n        ratio_test=0.80,\n        ransac_reproj=5.5,\n        split_large_at=9999,\n        preserve_clusters_up_to=9999,\n        max_cluster_pair_size=40,\n        conservative=EdgeRule(\"conservative\", min_inliers=18, min_inlier_ratio=0.18, min_score=0.36),\n        balanced=EdgeRule(\"balanced\", min_inliers=11, min_inlier_ratio=0.13, min_score=0.25),\n        swing=EdgeRule(\"swing\", min_inliers=7, min_inlier_ratio=0.09, min_score=0.18),\n    ),\n    \"SeaTurtleID2022\": SpeciesConfig(\n        species=\"SeaTurtleID2022\",\n        roi_kind=\"turtle_head_scutes\",\n        target_size=(416, 416),\n        top_k=22,\n        max_keypoints=900,\n        ratio_test=0.76,\n        ransac_reproj=4.5,\n        split_large_at=9999,\n        preserve_clusters_up_to=9999,\n        max_cluster_pair_size=35,\n        conservative=EdgeRule(\"conservative\", min_inliers=28, min_inlier_ratio=0.24, min_score=0.47),\n        balanced=EdgeRule(\"balanced\", min_inliers=22, min_inlier_ratio=0.20, min_score=0.39),\n        swing=EdgeRule(\"swing\", min_inliers=17, min_inlier_ratio=0.15, min_score=0.32),\n    ),\n    \"TexasHornedLizards\": SpeciesConfig(\n        species=\"TexasHornedLizards\",\n        roi_kind=\"texas_ventral_dots\",\n        target_size=(448, 320),\n        top_k=74,\n        max_keypoints=900,\n        ratio_test=0.78,\n        ransac_reproj=5.0,\n        split_large_at=20,\n        preserve_clusters_up_to=10,\n        max_cluster_pair_size=60,\n        conservative=EdgeRule(\"conservative\", min_inliers=18, min_inlier_ratio=0.26, min_score=0.34),\n        balanced=EdgeRule(\"balanced\", min_inliers=12, min_inlier_ratio=0.20, min_score=0.25),\n        swing=EdgeRule(\"swing\", min_inliers=8, min_inlier_ratio=0.16, min_score=0.18),\n    ),\n}\n\n\nOPTIONAL_SUBMISSION_FILENAMES = {\n    \"current_best\": CURRENT_BEST_FILENAME,\n    \"arwildfusion_v3_rescue_v2\": \"submission_sam_arwildfusion_v3_rescue_lbshape_v2.csv\",\n    \"p06_miewid_mega\": \"submission_p06_miewid_plus_mega_l384.csv\",\n    \"dualfoundation_species_hybrid\": \"submission_species_hybrid_v20260425.csv\",\n    \"dualfoundation_selected\": \"submission_selected_specieswise_hybrid.csv\",\n    \"specieswise_best_localari\": \"submission_specieswise_best_localari_hybrid_v20260425.csv\",\n}\n\n\nclass UnionFind:\n    def __init__(self, values: Iterable):\n        self.parent = {v: v for v in values}\n        self.size = {v: 1 for v in values}\n\n    def find(self, value):\n        if value not in self.parent:\n            self.parent[value] = value\n            self.size[value] = 1\n            return value\n        root = value\n        while self.parent[root] != root:\n            root = self.parent[root]\n        while self.parent[value] != value:\n            nxt = self.parent[value]\n            self.parent[value] = root\n            value = nxt\n        return root\n\n    def union(self, a, b):\n        ra = self.find(a)\n        rb = self.find(b)\n        if ra == rb:\n            return ra\n        if self.size[ra] < self.size[rb]:\n            ra, rb = rb, ra\n        self.parent[rb] = ra\n        self.size[ra] += self.size[rb]\n        return ra\n\n\n@dataclass\nclass PatternItem:\n    image_id: int\n    row_idx: int\n    species: str\n    orientation: str\n    view_path: str\n    view_source: str\n    quality: float\n    keypoints: np.ndarray\n    descriptors: np.ndarray | None\n    vector: np.ndarray\n    fallback_vector: np.ndarray\n    part_vectors: np.ndarray\n    part_visibility: np.ndarray\n    fallback_part_vectors: np.ndarray\n    fallback_part_visibility: np.ndarray\n    visibility_coverage: float\n    n_keypoints: int\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=(\n            \"Final-swing species-specific fingerprint ensemble for AnimalCLEF2026. \"\n            \"Uses saved SAM views when present, extracts different local pattern ROIs \"\n            \"per species, verifies shortlisted pairs geometrically, and writes \"\n            \"conservative/balanced/high-swing hybrid submissions over the current best.\"\n        )\n    )\n    parser.add_argument(\"--data-root\", type=str, default=None)\n    parser.add_argument(\"--output-root\", type=str, default=None)\n    parser.add_argument(\"--sam-manifest\", type=str, default=None)\n    parser.add_argument(\"--current-best-submission\", type=str, default=None)\n    parser.add_argument(\"--extra-submission-root\", action=\"append\", default=[])\n    parser.add_argument(\"--species\", nargs=\"*\", default=SPECIES_ORDER)\n    parser.add_argument(\"--max-images-per-species\", type=int, default=0)\n    parser.add_argument(\"--max-side\", type=int, default=820)\n    parser.add_argument(\"--top-k-scale\", type=float, default=1.0)\n    parser.add_argument(\"--pair-budget-per-species\", type=int, default=65000)\n    parser.add_argument(\"--save-roi-preview\", action=\"store_true\")\n    parser.add_argument(\"--smoke\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef find_data_root(user_value: str | None) -> Path:\n    candidates: list[Path] = []\n    if user_value:\n        candidates.append(Path(user_value))\n    candidates.extend(\n        [\n            Path(\"/kaggle/input/animal-clef-2026\"),\n            Path(\"/kaggle/input/competitions/animal-clef-2026\"),\n            Path.cwd() / \"animal-clef-2026\",\n            Path.cwd().parent / \"animal-clef-2026\",\n        ]\n    )\n    for root in candidates:\n        if (root / \"metadata.csv\").exists() and (root / \"sample_submission.csv\").exists():\n            return root.resolve()\n    raise FileNotFoundError(\"Could not locate AnimalCLEF2026 data root.\")\n\n\ndef find_sam_manifest(user_value: str | None) -> Path | None:\n    if user_value:\n        p = Path(user_value)\n        if p.exists():\n            return p.resolve()\n    direct = [\n        Path(\"/kaggle/working/animalclef_sam3_views_cache/reports/view_manifest_sam3_all_species.csv\"),\n        Path(\"/kaggle/input/animalclef2026-export-sam3-views-all-species-v2026/animalclef_sam3_views_cache/reports/view_manifest_sam3_all_species.csv\"),\n        Path(\"/kaggle/input/animalclef2026-export-sam3-views-all-species-v2026/reports/view_manifest_sam3_all_species.csv\"),\n        Path.cwd() / \"animalclef_sam3_views_cache\" / \"reports\" / \"view_manifest_sam3_all_species.csv\",\n        Path.cwd().parent / \"animalclef_sam3_views_cache\" / \"reports\" / \"view_manifest_sam3_all_species.csv\",\n    ]\n    for p in direct:\n        if p.exists():\n            return p.resolve()\n    for base in [Path(\"/kaggle/input\"), Path.cwd(), Path.cwd().parent]:\n        if base.exists():\n            try:\n                matches = list(base.rglob(\"view_manifest_sam3_all_species.csv\"))\n            except Exception:\n                matches = []\n            if matches:\n                return matches[0].resolve()\n    return None\n\n\ndef find_file_everywhere(filename: str, roots: Iterable[Path]) -> Path | None:\n    for root in roots:\n        candidate = root / filename\n        if candidate.exists():\n            return candidate.resolve()\n    for root in roots:\n        if not root.exists():\n            continue\n        try:\n            matches = list(root.rglob(filename))\n        except Exception:\n            matches = []\n        if matches:\n            matches.sort(key=lambda p: len(str(p)))\n            return matches[0].resolve()\n    return None\n\n\ndef find_submission_sources(args: argparse.Namespace, data_root: Path) -> dict[str, Path]:\n    roots: list[Path] = [\n        Path.cwd(),\n        Path.cwd().parent,\n        data_root.parent,\n        Path(\"/kaggle/input\"),\n    ]\n    roots.extend(Path(p) for p in args.extra_submission_root)\n    local_project = Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\")\n    roots.extend(\n        [\n            local_project / \"current_wildfusion_graph_v20260423\",\n            local_project / \"AnimalCLEF2026 SAM ARWildFusion v3_update\",\n            local_project / \"AnimalCLEF2026 SAM ARWildFusion v3_update\" / \"submissions\",\n            local_project / \"archive\" / \"AnimalCLEF2026 v4 Backbone Sweep + Non-EfficientNet Candidates\",\n            local_project / \"Masked Dual-Foundation Search v20260425 output\",\n        ]\n    )\n    roots = [p for p in roots if p is not None]\n\n    found: dict[str, Path] = {}\n    if args.current_best_submission:\n        p = Path(args.current_best_submission)\n        if p.exists():\n            found[\"current_best\"] = p.resolve()\n\n    for label, filename in OPTIONAL_SUBMISSION_FILENAMES.items():\n        if label in found:\n            continue\n        path = find_file_everywhere(filename, roots)\n        if path is not None:\n            found[label] = path\n\n    if \"current_best\" not in found:\n        raise FileNotFoundError(f\"Could not find {CURRENT_BEST_FILENAME}.\")\n    return found\n\n\ndef export_root_from_manifest(manifest_path: Path) -> Path:\n    if manifest_path.parent.name == \"reports\":\n        return manifest_path.parent.parent\n    return manifest_path.parent\n\n\ndef remap_export_path(path_value, export_root: Path | None) -> Path | None:\n    if path_value is None or pd.isna(path_value):\n        return None\n    s = str(path_value).strip()\n    if not s:\n        return None\n    p = Path(s)\n    if p.exists():\n        return p.resolve()\n    if export_root is None:\n        return None\n    normalized = s.replace(\"\\\\\", \"/\")\n    markers = [\n        \"animalclef_sam3_views_cache/\",\n        \"views/\",\n        \"mask_loose_square/\",\n        \"mask_full_square/\",\n    ]\n    for marker in markers:\n        if marker in normalized:\n            rel = normalized.split(marker, 1)[1]\n            if marker != \"animalclef_sam3_views_cache/\":\n                rel = marker + rel\n            candidate = export_root / Path(rel)\n            if candidate.exists():\n                return candidate.resolve()\n    return None\n\n\ndef prepare_test_metadata(data_root: Path, sam_manifest: Path | None) -> tuple[pd.DataFrame, dict]:\n    metadata = pd.read_csv(data_root / \"metadata.csv\").reset_index(drop=True)\n    if \"row_idx\" not in metadata.columns:\n        metadata[\"row_idx\"] = np.arange(len(metadata), dtype=np.int64)\n    if \"dataset\" in metadata.columns:\n        metadata[\"species_id\"] = metadata[\"dataset\"].astype(str)\n    else:\n        metadata[\"species_id\"] = metadata[\"path\"].str.replace(\"\\\\\", \"/\", regex=False).str.split(\"/\").str[1]\n    if \"split\" not in metadata.columns:\n        metadata[\"split\"] = np.where(metadata[\"path\"].str.contains(\"/test/\"), \"test\", \"train\")\n    if \"orientation\" not in metadata.columns:\n        metadata[\"orientation\"] = \"unknown\"\n    metadata[\"source_path\"] = metadata[\"path\"].map(lambda p: str(data_root / str(p)))\n\n    test = metadata[metadata[\"split\"].eq(\"test\")].copy()\n    test[\"view_path\"] = test[\"source_path\"]\n    test[\"view_source\"] = \"original\"\n    test[\"mask_path\"] = \"\"\n    test[\"mask_ok\"] = False\n\n    info = {\"manifest_path\": str(sam_manifest) if sam_manifest else None, \"manifest_rows\": 0, \"resolved_views\": 0}\n    if sam_manifest is None:\n        return test, info\n\n    manifest = pd.read_csv(sam_manifest)\n    info[\"manifest_rows\"] = int(len(manifest))\n    export_root = export_root_from_manifest(sam_manifest)\n    merge_key = \"row_idx\" if \"row_idx\" in manifest.columns else \"image_id\" if \"image_id\" in manifest.columns else None\n    if merge_key is None:\n        return test, info\n    merged = test.merge(manifest, on=merge_key, how=\"left\", suffixes=(\"\", \"_sam\"))\n\n    view_paths: list[str] = []\n    sam_view_paths: list[str] = []\n    mask_paths: list[str] = []\n    view_sources: list[str] = []\n    mask_ok: list[bool] = []\n    for row in merged.to_dict(\"records\"):\n        resolved_sam_view = None\n        for col in [\"loose_path\", \"mask_loose_path\", \"mask_full_path\", \"view_path\"]:\n            if col in row:\n                resolved_sam_view = remap_export_path(row.get(col), export_root)\n                if resolved_sam_view is not None:\n                    break\n        resolved_mask = None\n        for col in [\"mask_path\", \"binary_mask_path\"]:\n            if col in row:\n                resolved_mask = remap_export_path(row.get(col), export_root)\n                if resolved_mask is not None:\n                    break\n        # Corrected assumption: species is known and each image contains one\n        # individual, so the original image remains the pattern source. SAM is\n        # a guide for animal/body visibility, not the image to compare.\n        view_paths.append(str(Path(row[\"source_path\"])))\n        sam_view_paths.append(str(resolved_sam_view) if resolved_sam_view else \"\")\n        view_sources.append(\"original_sam_guided\" if resolved_mask is not None else \"original\")\n        mask_paths.append(str(resolved_mask) if resolved_mask else \"\")\n        mask_ok.append(resolved_mask is not None)\n    merged[\"view_path\"] = view_paths\n    merged[\"sam_view_path\"] = sam_view_paths\n    merged[\"view_source\"] = view_sources\n    merged[\"mask_path\"] = mask_paths\n    merged[\"mask_ok\"] = mask_ok\n    info[\"resolved_views\"] = int(sum(mask_ok))\n    return merged, info\n\n\ndef read_rgb(path: str | Path, max_side: int) -> np.ndarray:\n    img = Image.open(path).convert(\"RGB\")\n    w, h = img.size\n    scale = min(1.0, float(max_side) / float(max(w, h)))\n    if scale < 1.0:\n        img = img.resize((max(1, int(round(w * scale))), max(1, int(round(h * scale)))), Image.Resampling.BILINEAR)\n    return np.asarray(img)\n\n\ndef read_rgb_with_optional_mask(\n    image_path: str | Path,\n    mask_path: str | Path | None,\n    max_side: int,\n) -> tuple[np.ndarray, np.ndarray | None]:\n    img = Image.open(image_path).convert(\"RGB\")\n    w, h = img.size\n    mask_img = None\n    if mask_path:\n        p = Path(mask_path)\n        if p.exists():\n            try:\n                mask_img = Image.open(p).convert(\"L\")\n                if mask_img.size != img.size:\n                    mask_img = mask_img.resize(img.size, Image.Resampling.NEAREST)\n            except Exception:\n                mask_img = None\n    scale = min(1.0, float(max_side) / float(max(w, h)))\n    if scale < 1.0:\n        new_size = (max(1, int(round(w * scale))), max(1, int(round(h * scale))))\n        img = img.resize(new_size, Image.Resampling.BILINEAR)\n        if mask_img is not None:\n            mask_img = mask_img.resize(new_size, Image.Resampling.NEAREST)\n    rgb = np.asarray(img)\n    mask = None\n    if mask_img is not None:\n        mask = np.where(np.asarray(mask_img) > 127, 255, 0).astype(np.uint8)\n        if float(mask.mean() / 255.0) < 0.01:\n            mask = None\n    return rgb, mask\n\n\ndef estimate_foreground_mask(rgb: np.ndarray) -> np.ndarray:\n    arr = rgb.astype(np.float32)\n    h, w = arr.shape[:2]\n    border = np.concatenate(\n        [\n            arr[: max(2, h // 40), :, :].reshape(-1, 3),\n            arr[-max(2, h // 40) :, :, :].reshape(-1, 3),\n            arr[:, : max(2, w // 40), :].reshape(-1, 3),\n            arr[:, -max(2, w // 40) :, :].reshape(-1, 3),\n        ],\n        axis=0,\n    )\n    border_rgb = np.median(border, axis=0)\n    diff_border = np.linalg.norm(arr - border_rgb[None, None, :], axis=2)\n    diff_export_bg = np.linalg.norm(arr - BACKGROUND_RGB[None, None, :], axis=2)\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    sat = hsv[:, :, 1].astype(np.float32)\n    mask = ((diff_export_bg > 20) & (diff_border > 8)) | ((diff_border > 28) & (sat > 18))\n    mask = mask.astype(np.uint8) * 255\n    kernel = np.ones((5, 5), np.uint8)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)\n    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)\n    if n_labels > 1:\n        biggest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))\n        mask = np.where(labels == biggest, 255, 0).astype(np.uint8)\n    coverage = float(mask.mean() / 255.0)\n    if coverage < 0.04 or coverage > 0.96:\n        mask = np.full((h, w), 255, dtype=np.uint8)\n    return mask\n\n\ndef bbox_from_mask(mask: np.ndarray, pad_frac: float = 0.05) -> tuple[int, int, int, int]:\n    ys, xs = np.where(mask > 0)\n    h, w = mask.shape[:2]\n    if len(xs) == 0:\n        return 0, 0, w, h\n    x1, x2 = int(xs.min()), int(xs.max()) + 1\n    y1, y2 = int(ys.min()), int(ys.max()) + 1\n    pad = int(round(max(x2 - x1, y2 - y1) * pad_frac))\n    return max(0, x1 - pad), max(0, y1 - pad), min(w, x2 + pad), min(h, y2 + pad)\n\n\ndef crop_to_mask(rgb: np.ndarray, mask: np.ndarray, pad_frac: float = 0.05) -> tuple[np.ndarray, np.ndarray]:\n    x1, y1, x2, y2 = bbox_from_mask(mask, pad_frac)\n    return rgb[y1:y2, x1:x2].copy(), mask[y1:y2, x1:x2].copy()\n\n\ndef pca_angle_degrees(mask: np.ndarray) -> float:\n    ys, xs = np.where(mask > 0)\n    if len(xs) < 30:\n        return 0.0\n    pts = np.stack([xs.astype(np.float32), ys.astype(np.float32)], axis=1)\n    pts -= pts.mean(axis=0, keepdims=True)\n    cov = np.cov(pts.T)\n    vals, vecs = np.linalg.eigh(cov)\n    vec = vecs[:, int(np.argmax(vals))]\n    return float(math.degrees(math.atan2(vec[1], vec[0])))\n\n\ndef rotate_bound(rgb: np.ndarray, mask: np.ndarray, angle: float) -> tuple[np.ndarray, np.ndarray]:\n    h, w = rgb.shape[:2]\n    center = (w / 2.0, h / 2.0)\n    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)\n    cos = abs(matrix[0, 0])\n    sin = abs(matrix[0, 1])\n    new_w = int((h * sin) + (w * cos))\n    new_h = int((h * cos) + (w * sin))\n    matrix[0, 2] += (new_w / 2.0) - center[0]\n    matrix[1, 2] += (new_h / 2.0) - center[1]\n    rgb_rot = cv2.warpAffine(\n        rgb,\n        matrix,\n        (new_w, new_h),\n        flags=cv2.INTER_LINEAR,\n        borderMode=cv2.BORDER_CONSTANT,\n        borderValue=(238, 238, 232),\n    )\n    mask_rot = cv2.warpAffine(\n        mask,\n        matrix,\n        (new_w, new_h),\n        flags=cv2.INTER_NEAREST,\n        borderMode=cv2.BORDER_CONSTANT,\n        borderValue=0,\n    )\n    return rgb_rot, mask_rot\n\n\ndef align_and_crop(rgb: np.ndarray, mask: np.ndarray, do_pca: bool = True) -> tuple[np.ndarray, np.ndarray]:\n    crop_rgb, crop_mask = crop_to_mask(rgb, mask, 0.08)\n    if do_pca:\n        angle = pca_angle_degrees(crop_mask)\n        if abs(angle) > 2:\n            crop_rgb, crop_mask = rotate_bound(crop_rgb, crop_mask, -angle)\n            crop_rgb, crop_mask = crop_to_mask(crop_rgb, crop_mask, 0.05)\n    return crop_rgb, crop_mask\n\n\ndef slice_roi(rgb: np.ndarray, mask: np.ndarray, box: tuple[float, float, float, float]) -> tuple[np.ndarray, np.ndarray]:\n    h, w = rgb.shape[:2]\n    x1 = int(round(w * box[0]))\n    y1 = int(round(h * box[1]))\n    x2 = int(round(w * box[2]))\n    y2 = int(round(h * box[3]))\n    x1, y1 = max(0, x1), max(0, y1)\n    x2, y2 = min(w, max(x1 + 1, x2)), min(h, max(y1 + 1, y2))\n    return rgb[y1:y2, x1:x2].copy(), mask[y1:y2, x1:x2].copy()\n\n\ndef resize_roi(rgb: np.ndarray, mask: np.ndarray, size: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:\n    w, h = size\n    rgb_resized = cv2.resize(rgb, (w, h), interpolation=cv2.INTER_AREA)\n    mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)\n    mask_resized = np.where(mask_resized > 0, 255, 0).astype(np.uint8)\n    return rgb_resized, mask_resized\n\n\ndef species_roi(\n    rgb: np.ndarray,\n    species: str,\n    orientation: str,\n    config: SpeciesConfig,\n    mask_override: np.ndarray | None = None,\n) -> tuple[np.ndarray, np.ndarray, float]:\n    mask = mask_override if mask_override is not None else estimate_foreground_mask(rgb)\n    if mask.shape[:2] != rgb.shape[:2]:\n        mask = cv2.resize(mask, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)\n        mask = np.where(mask > 0, 255, 0).astype(np.uint8)\n    quality = 1.0\n    orientation = (orientation or \"unknown\").lower()\n    if config.roi_kind == \"turtle_head_scutes\":\n        aligned_rgb, aligned_mask = align_and_crop(rgb, mask, do_pca=False)\n        roi_rgb, roi_mask = slice_roi(aligned_rgb, aligned_mask, (0.04, 0.02, 0.96, 0.96))\n    else:\n        aligned_rgb, aligned_mask = align_and_crop(rgb, mask, do_pca=True)\n        if config.roi_kind == \"lynx_flank_spots\":\n            if orientation in {\"front\", \"back\", \"unknown\", \"\"}:\n                quality *= 0.72\n            roi_rgb, roi_mask = slice_roi(aligned_rgb, aligned_mask, (0.08, 0.14, 0.94, 0.88))\n        elif config.roi_kind == \"salamander_straight_strip\":\n            roi_rgb, roi_mask = slice_roi(aligned_rgb, aligned_mask, (0.03, 0.10, 0.97, 0.90))\n        elif config.roi_kind == \"texas_ventral_dots\":\n            roi_rgb, roi_mask = slice_roi(aligned_rgb, aligned_mask, (0.10, 0.08, 0.90, 0.94))\n        else:\n            roi_rgb, roi_mask = aligned_rgb, aligned_mask\n\n    roi_rgb, roi_mask = resize_roi(roi_rgb, roi_mask, config.target_size)\n    if config.roi_kind == \"texas_ventral_dots\":\n        h, w = roi_mask.shape[:2]\n        ellipse = np.zeros((h, w), dtype=np.uint8)\n        cv2.ellipse(\n            ellipse,\n            (w // 2, int(h * 0.54)),\n            (int(w * 0.38), int(h * 0.38)),\n            0,\n            0,\n            360,\n            255,\n            -1,\n        )\n        roi_mask = cv2.bitwise_and(roi_mask, ellipse)\n    if float(roi_mask.mean() / 255.0) < 0.05:\n        roi_mask = np.full(roi_mask.shape, 255, dtype=np.uint8)\n        quality *= 0.70\n    return roi_rgb, roi_mask, quality\n\n\ndef pattern_gray(rgb: np.ndarray, species: str) -> np.ndarray:\n    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    l = lab[:, :, 0]\n    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))\n    l_eq = clahe.apply(l)\n    if species in {\"LynxID2025\", \"TexasHornedLizards\"}:\n        dark = 255 - l_eq\n        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))\n        blackhat = cv2.morphologyEx(l_eq, cv2.MORPH_BLACKHAT, kernel)\n        gray = cv2.addWeighted(dark, 0.62, blackhat, 0.38, 0)\n    elif species == \"SalamanderID2025\":\n        yellow = lab[:, :, 2]\n        sat = hsv[:, :, 1]\n        gray = cv2.addWeighted(clahe.apply(yellow), 0.55, clahe.apply(sat), 0.45, 0)\n    else:\n        a = clahe.apply(lab[:, :, 1])\n        b = clahe.apply(lab[:, :, 2])\n        gray = cv2.addWeighted(l_eq, 0.60, cv2.addWeighted(a, 0.5, b, 0.5, 0), 0.40, 0)\n    gray = cv2.GaussianBlur(gray, (3, 3), 0)\n    return gray.astype(np.uint8)\n\n\ndef create_detector(max_keypoints: int):\n    try:\n        return \"sift\", cv2.SIFT_create(nfeatures=max_keypoints, contrastThreshold=0.015, edgeThreshold=12)\n    except Exception:\n        return \"orb\", cv2.ORB_create(nfeatures=max_keypoints, fastThreshold=7)\n\n\ndef normalize_vector(vec: np.ndarray) -> np.ndarray:\n    vec = vec.astype(np.float32, copy=False)\n    norm = float(np.linalg.norm(vec))\n    if norm > 0:\n        vec = vec / norm\n    return vec.astype(np.float32, copy=False)\n\n\ndef part_grid(config: SpeciesConfig) -> tuple[int, int]:\n    if config.roi_kind == \"salamander_straight_strip\":\n        return 3, 10\n    if config.roi_kind == \"turtle_head_scutes\":\n        return 4, 4\n    if config.roi_kind == \"texas_ventral_dots\":\n        return 5, 7\n    return 4, 7\n\n\ndef neutralize_missing(gray: np.ndarray, mask: np.ndarray | None) -> np.ndarray:\n    if mask is None:\n        return gray\n    valid = mask > 0\n    if valid.mean() < 0.02:\n        return gray\n    neutral = gray.copy()\n    fill = int(np.median(gray[valid]))\n    neutral[~valid] = fill\n    return neutral\n\n\ndef compute_vector(rgb: np.ndarray, gray: np.ndarray, mask: np.ndarray | None = None) -> np.ndarray:\n    gray_for_thumb = neutralize_missing(gray, mask)\n    small = cv2.resize(gray_for_thumb, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32).reshape(-1) / 255.0\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    hist_parts = []\n    for channel, bins, limit in [(0, 24, 180), (1, 16, 256), (2, 16, 256)]:\n        hist_mask = mask if mask is not None and float(mask.mean()) > 0 else None\n        hist = cv2.calcHist([hsv], [channel], hist_mask, [bins], [0, limit]).astype(np.float32).reshape(-1)\n        hist /= max(1e-6, float(hist.sum()))\n        hist_parts.append(hist)\n    vec = np.concatenate([small, *hist_parts]).astype(np.float32)\n    return normalize_vector(vec)\n\n\ndef compute_part_vectors(\n    rgb: np.ndarray,\n    gray: np.ndarray,\n    mask: np.ndarray,\n    config: SpeciesConfig,\n) -> tuple[np.ndarray, np.ndarray, float]:\n    rows, cols = part_grid(config)\n    h, w = gray.shape[:2]\n    vectors: list[np.ndarray] = []\n    visibility: list[bool] = []\n    total_valid = float((mask > 0).mean())\n    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)\n    for gy in range(rows):\n        y1 = int(round(h * gy / rows))\n        y2 = int(round(h * (gy + 1) / rows))\n        for gx in range(cols):\n            x1 = int(round(w * gx / cols))\n            x2 = int(round(w * (gx + 1) / cols))\n            cell_mask = mask[y1:y2, x1:x2]\n            cell_gray = gray[y1:y2, x1:x2]\n            cell_rgb = rgb[y1:y2, x1:x2]\n            cell_hsv = hsv[y1:y2, x1:x2]\n            coverage = float((cell_mask > 0).mean()) if cell_mask.size else 0.0\n            visible = coverage >= 0.12\n            visibility.append(visible)\n            if not visible:\n                vectors.append(np.zeros(64 + 32, dtype=np.float32))\n                continue\n            valid = cell_mask > 0\n            cell_neutral = cell_gray.copy()\n            cell_neutral[~valid] = int(np.median(cell_gray[valid]))\n            thumb = cv2.resize(cell_neutral, (8, 8), interpolation=cv2.INTER_AREA).astype(np.float32).reshape(-1) / 255.0\n            hist_parts = []\n            for channel, bins, limit in [(0, 12, 180), (1, 10, 256), (2, 10, 256)]:\n                hist = cv2.calcHist([cell_hsv], [channel], cell_mask, [bins], [0, limit]).astype(np.float32).reshape(-1)\n                hist /= max(1e-6, float(hist.sum()))\n                hist_parts.append(hist)\n            vectors.append(normalize_vector(np.concatenate([thumb, *hist_parts]).astype(np.float32)))\n    return np.stack(vectors).astype(np.float32), np.asarray(visibility, dtype=bool), total_valid\n\n\ndef part_similarity_from_arrays(\n    vec_a: np.ndarray,\n    vis_a: np.ndarray,\n    vec_b: np.ndarray,\n    vis_b: np.ndarray,\n) -> tuple[float, float, int]:\n    if vec_a.size == 0 or vec_b.size == 0:\n        return 0.0, 0.0, 0\n    common = vis_a & vis_b\n    n_common = int(common.sum())\n    n_possible = int((vis_a | vis_b).sum())\n    if n_common == 0:\n        return 0.0, 0.0, 0\n    sims = np.sum(vec_a[common] * vec_b[common], axis=1)\n    sims = np.clip(sims, -1.0, 1.0)\n    if len(sims) >= 4:\n        # Partial matching: emphasize the best mutually visible zones instead\n        # of punishing a crop where SAM removed a limb/body patch.\n        keep = max(2, int(math.ceil(len(sims) * 0.70)))\n        score = float(np.mean(np.sort(sims)[-keep:]))\n    else:\n        score = float(np.mean(sims))\n    overlap = float(n_common / max(1, n_possible))\n    return score, overlap, n_common\n\n\ndef occlusion_aware_similarity(a: PatternItem, b: PatternItem) -> tuple[float, float, int, float]:\n    candidates = [\n        part_similarity_from_arrays(a.part_vectors, a.part_visibility, b.part_vectors, b.part_visibility),\n        part_similarity_from_arrays(\n            a.fallback_part_vectors,\n            a.fallback_part_visibility,\n            b.fallback_part_vectors,\n            b.fallback_part_visibility,\n        ),\n        part_similarity_from_arrays(a.part_vectors, a.part_visibility, b.fallback_part_vectors, b.fallback_part_visibility),\n        part_similarity_from_arrays(a.fallback_part_vectors, a.fallback_part_visibility, b.part_vectors, b.part_visibility),\n    ]\n    part_score, part_overlap, part_cells = max(candidates, key=lambda x: (x[0], x[2]))\n    global_primary = float(np.dot(a.vector, b.vector))\n    global_fallback = float(np.dot(a.fallback_vector, b.fallback_vector))\n    visual_sim = max(global_primary, global_fallback, part_score)\n    return visual_sim, float(part_score), int(part_cells), float(part_overlap)\n\n\ndef extract_pattern_item(row: dict, config: SpeciesConfig, max_side: int, detector_name: str, detector) -> PatternItem:\n    mask_path = str(row.get(\"mask_path\", \"\")).strip()\n    rgb, sam_mask = read_rgb_with_optional_mask(row[\"view_path\"], mask_path, max_side=max_side)\n    roi_rgb, roi_mask, quality = species_roi(\n        rgb,\n        row[\"species_id\"],\n        str(row.get(\"orientation\", \"unknown\")),\n        config,\n        mask_override=sam_mask,\n    )\n    gray = pattern_gray(roi_rgb, row[\"species_id\"])\n    kps, desc = detector.detectAndCompute(gray, roi_mask)\n    if kps is None:\n        kps = []\n    pts = np.array([kp.pt for kp in kps], dtype=np.float32).reshape(-1, 2)\n    if desc is not None and len(desc) > 0:\n        if detector_name == \"sift\":\n            desc = desc.astype(np.float32)\n            desc /= np.maximum(1e-7, desc.sum(axis=1, keepdims=True))\n            desc = np.sqrt(desc)\n        else:\n            desc = desc.astype(np.uint8)\n    else:\n        desc = None\n    vec = compute_vector(roi_rgb, gray, roi_mask)\n    part_vecs, part_vis, coverage = compute_part_vectors(roi_rgb, gray, roi_mask, config)\n    fallback_vec = vec\n    fallback_part_vecs = part_vecs\n    fallback_part_vis = part_vis\n    source_path = str(row.get(\"source_path\", \"\")).strip()\n    # Fallback descriptor deliberately ignores SAM if present. It helps\n    # shortlist candidates when SAM masks are too conservative, while the\n    # primary descriptor still uses the SAM-guided animal/body region.\n    if source_path and Path(source_path).exists() and sam_mask is not None:\n        try:\n            original_rgb = read_rgb(source_path, max_side=max_side)\n            original_roi_rgb, original_roi_mask, original_quality = species_roi(\n                original_rgb,\n                row[\"species_id\"],\n                str(row.get(\"orientation\", \"unknown\")),\n                config,\n                mask_override=None,\n            )\n            original_gray = pattern_gray(original_roi_rgb, row[\"species_id\"])\n            fallback_vec = compute_vector(original_roi_rgb, original_gray, original_roi_mask)\n            fallback_part_vecs, fallback_part_vis, original_coverage = compute_part_vectors(\n                original_roi_rgb,\n                original_gray,\n                original_roi_mask,\n                config,\n            )\n            quality = min(quality, 0.75 + 0.25 * original_quality)\n            coverage = max(coverage, min(0.95, original_coverage))\n        except Exception:\n            fallback_vec = vec\n            fallback_part_vecs = part_vecs\n            fallback_part_vis = part_vis\n    if coverage < 0.22 and row[\"species_id\"] != \"SeaTurtleID2022\":\n        quality *= 0.78\n    combined_vec = normalize_vector(0.72 * vec + 0.28 * fallback_vec)\n    return PatternItem(\n        image_id=int(row[\"image_id\"]),\n        row_idx=int(row.get(\"row_idx\", row[\"image_id\"])),\n        species=str(row[\"species_id\"]),\n        orientation=str(row.get(\"orientation\", \"unknown\")).lower(),\n        view_path=str(row[\"view_path\"]),\n        view_source=str(row.get(\"view_source\", \"original\")),\n        quality=float(quality),\n        keypoints=pts,\n        descriptors=desc,\n        vector=combined_vec,\n        fallback_vector=fallback_vec,\n        part_vectors=part_vecs,\n        part_visibility=part_vis,\n        fallback_part_vectors=fallback_part_vecs,\n        fallback_part_visibility=fallback_part_vis,\n        visibility_coverage=float(coverage),\n        n_keypoints=len(kps),\n    )\n\n\ndef save_roi_preview(rows: list[dict], out_path: Path, config: SpeciesConfig, max_side: int, limit: int = 24) -> None:\n    if not rows:\n        return\n    thumbs = []\n    for row in rows[:limit]:\n        try:\n            mask_path = str(row.get(\"mask_path\", \"\")).strip()\n            rgb, sam_mask = read_rgb_with_optional_mask(row[\"view_path\"], mask_path, max_side=max_side)\n            roi_rgb, roi_mask, _ = species_roi(\n                rgb,\n                row[\"species_id\"],\n                str(row.get(\"orientation\", \"unknown\")),\n                config,\n                mask_override=sam_mask,\n            )\n            overlay = roi_rgb.copy()\n            overlay[roi_mask == 0] = (overlay[roi_mask == 0] * 0.35 + np.array([238, 238, 232]) * 0.65).astype(np.uint8)\n            thumbs.append(Image.fromarray(overlay).resize((160, 112)))\n        except Exception:\n            continue\n    if not thumbs:\n        return\n    cols = 4\n    rows_n = int(math.ceil(len(thumbs) / cols))\n    canvas = Image.new(\"RGB\", (cols * 160, rows_n * 112), (238, 238, 232))\n    for idx, thumb in enumerate(thumbs):\n        canvas.paste(thumb, ((idx % cols) * 160, (idx // cols) * 112))\n    canvas.save(out_path, quality=90)\n\n\ndef load_submission(path: Path) -> pd.DataFrame:\n    df = pd.read_csv(path)\n    if \"image_id\" not in df.columns or \"cluster\" not in df.columns:\n        raise ValueError(f\"{path} must contain image_id and cluster columns.\")\n    df = df[[\"image_id\", \"cluster\"]].copy()\n    df[\"image_id\"] = df[\"image_id\"].astype(int)\n    df[\"cluster\"] = df[\"cluster\"].astype(str)\n    return df\n\n\ndef labels_for_species(submission: pd.DataFrame, species_rows: pd.DataFrame) -> dict[int, str]:\n    merged = species_rows[[\"image_id\"]].merge(submission, on=\"image_id\", how=\"left\")\n    if merged[\"cluster\"].isna().any():\n        missing = merged.loc[merged[\"cluster\"].isna(), \"image_id\"].head().tolist()\n        raise ValueError(f\"Submission is missing image ids: {missing}\")\n    return {int(r.image_id): str(r.cluster) for r in merged.itertuples(index=False)}\n\n\ndef pair_key(a: int, b: int) -> tuple[int, int]:\n    return (a, b) if a < b else (b, a)\n\n\ndef cluster_pair_votes(\n    source_submissions: dict[str, pd.DataFrame],\n    species_rows: pd.DataFrame,\n    config: SpeciesConfig,\n) -> dict[tuple[int, int], int]:\n    votes: dict[tuple[int, int], int] = {}\n    for source_name, sub in source_submissions.items():\n        if source_name == \"current_best\":\n            continue\n        labels = labels_for_species(sub, species_rows)\n        groups: dict[str, list[int]] = {}\n        for image_id, cluster in labels.items():\n            groups.setdefault(cluster, []).append(image_id)\n        for members in groups.values():\n            if 1 < len(members) <= config.max_cluster_pair_size:\n                for a, b in itertools.combinations(sorted(members), 2):\n                    key = pair_key(a, b)\n                    votes[key] = votes.get(key, 0) + 1\n    return votes\n\n\ndef current_cluster_pairs(labels: dict[int, str], max_cluster_pair_size: int) -> set[tuple[int, int]]:\n    groups: dict[str, list[int]] = {}\n    for image_id, cluster in labels.items():\n        groups.setdefault(cluster, []).append(image_id)\n    pairs: set[tuple[int, int]] = set()\n    for members in groups.values():\n        if 1 < len(members) <= max_cluster_pair_size:\n            for a, b in itertools.combinations(sorted(members), 2):\n                pairs.add(pair_key(a, b))\n    return pairs\n\n\ndef orientation_compatible(species: str, o1: str, o2: str, allow_opposite_lynx: bool = False) -> tuple[bool, str]:\n    o1 = (o1 or \"unknown\").lower()\n    o2 = (o2 or \"unknown\").lower()\n    if species != \"LynxID2025\":\n        return True, \"not_applicable\"\n    side_set = {\"left\", \"right\"}\n    if o1 in side_set and o2 in side_set and o1 != o2:\n        return allow_opposite_lynx, \"lynx_opposite_flank\"\n    if o1 in {\"front\", \"back\"} or o2 in {\"front\", \"back\"}:\n        return True, \"lynx_weak_pose\"\n    return True, \"lynx_same_or_unknown\"\n\n\ndef shortlist_pairs(\n    items: list[PatternItem],\n    current_labels: dict[int, str],\n    alt_votes: dict[tuple[int, int], int],\n    config: SpeciesConfig,\n    top_k_scale: float,\n    pair_budget: int,\n) -> set[tuple[int, int]]:\n    image_ids = [it.image_id for it in items]\n    item_by_id = {it.image_id: it for it in items}\n    pairs = set(current_cluster_pairs(current_labels, config.max_cluster_pair_size))\n    pairs.update(alt_votes.keys())\n\n    vectors = np.stack([it.vector for it in items]).astype(np.float32)\n    sim = vectors @ vectors.T\n    np.fill_diagonal(sim, -np.inf)\n    top_k = max(1, int(round(config.top_k * top_k_scale)))\n    for i, image_id in enumerate(image_ids):\n        k = min(top_k, len(image_ids) - 1)\n        if k <= 0:\n            continue\n        idx = np.argpartition(-sim[i], kth=k - 1)[:k]\n        for j in idx:\n            other_id = image_ids[int(j)]\n            if other_id == image_id:\n                continue\n            a, b = pair_key(image_id, other_id)\n            ok, _ = orientation_compatible(\n                config.species,\n                item_by_id[a].orientation,\n                item_by_id[b].orientation,\n                allow_opposite_lynx=False,\n            )\n            if ok or current_labels.get(a) == current_labels.get(b):\n                pairs.add((a, b))\n\n    if len(pairs) <= pair_budget:\n        return pairs\n\n    id_to_idx = {image_id: idx for idx, image_id in enumerate(image_ids)}\n    priority = []\n    for a, b in pairs:\n        ia = id_to_idx[a]\n        ib = id_to_idx[b]\n        same_current = current_labels.get(a) == current_labels.get(b)\n        vote = alt_votes.get((a, b), 0)\n        priority.append((same_current, vote, float(sim[ia, ib]), a, b))\n    priority.sort(reverse=True)\n    return {pair_key(a, b) for _, _, _, a, b in priority[:pair_budget]}\n\n\ndef bf_matcher(detector_name: str):\n    if detector_name == \"sift\":\n        return cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)\n    return cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)\n\n\ndef local_match_score(\n    a: PatternItem,\n    b: PatternItem,\n    config: SpeciesConfig,\n    detector_name: str,\n    ratio_test: float,\n) -> dict:\n    visual_sim, part_score, part_cells, part_overlap = occlusion_aware_similarity(a, b)\n    if a.descriptors is None or b.descriptors is None or len(a.descriptors) < 4 or len(b.descriptors) < 4:\n        return {\n            \"inliers\": 0,\n            \"good_matches\": 0,\n            \"inlier_ratio\": 0.0,\n            \"spatial_coverage\": 0.0,\n            \"local_score\": max(0.0, visual_sim) * 0.12,\n            \"visual_sim\": visual_sim,\n            \"part_score\": part_score,\n            \"part_overlap\": part_overlap,\n            \"part_cells\": part_cells,\n            \"coverage_a\": float(a.visibility_coverage),\n            \"coverage_b\": float(b.visibility_coverage),\n        }\n    matcher = bf_matcher(detector_name)\n    try:\n        knn = matcher.knnMatch(a.descriptors, b.descriptors, k=2)\n    except Exception:\n        return {\n            \"inliers\": 0,\n            \"good_matches\": 0,\n            \"inlier_ratio\": 0.0,\n            \"spatial_coverage\": 0.0,\n            \"local_score\": max(0.0, visual_sim) * 0.12,\n            \"visual_sim\": visual_sim,\n            \"part_score\": part_score,\n            \"part_overlap\": part_overlap,\n            \"part_cells\": part_cells,\n            \"coverage_a\": float(a.visibility_coverage),\n            \"coverage_b\": float(b.visibility_coverage),\n        }\n    good = []\n    for pair in knn:\n        if len(pair) < 2:\n            continue\n        m, n = pair\n        if m.distance < ratio_test * n.distance:\n            good.append(m)\n    if len(good) < 4:\n        return {\n            \"inliers\": 0,\n            \"good_matches\": len(good),\n            \"inlier_ratio\": 0.0,\n            \"spatial_coverage\": 0.0,\n            \"local_score\": max(0.0, visual_sim) * 0.12,\n            \"visual_sim\": visual_sim,\n            \"part_score\": part_score,\n            \"part_overlap\": part_overlap,\n            \"part_cells\": part_cells,\n            \"coverage_a\": float(a.visibility_coverage),\n            \"coverage_b\": float(b.visibility_coverage),\n        }\n    pts_a = np.float32([a.keypoints[m.queryIdx] for m in good]).reshape(-1, 2)\n    pts_b = np.float32([b.keypoints[m.trainIdx] for m in good]).reshape(-1, 2)\n    _, inlier_mask = cv2.estimateAffinePartial2D(\n        pts_a,\n        pts_b,\n        method=cv2.RANSAC,\n        ransacReprojThreshold=config.ransac_reproj,\n        maxIters=2000,\n        confidence=0.995,\n    )\n    if inlier_mask is None:\n        inliers = 0\n        inlier_flags = np.zeros(len(good), dtype=bool)\n    else:\n        inlier_flags = inlier_mask.reshape(-1).astype(bool)\n        inliers = int(inlier_flags.sum())\n    denom = max(1, min(len(a.keypoints), len(b.keypoints), len(good)))\n    inlier_ratio = float(inliers / denom)\n    if inliers > 1:\n        pa = pts_a[inlier_flags]\n        pb = pts_b[inlier_flags]\n        cov_a = (float(pa[:, 0].max() - pa[:, 0].min()) * float(pa[:, 1].max() - pa[:, 1].min())) / max(\n            1.0, float(config.target_size[0] * config.target_size[1])\n        )\n        cov_b = (float(pb[:, 0].max() - pb[:, 0].min()) * float(pb[:, 1].max() - pb[:, 1].min())) / max(\n            1.0, float(config.target_size[0] * config.target_size[1])\n        )\n        spatial_coverage = float(min(1.0, math.sqrt(max(0.0, cov_a * cov_b)) * 4.0))\n    else:\n        spatial_coverage = 0.0\n    inlier_term = min(1.0, inliers / max(1.0, config.conservative.min_inliers * 1.4))\n    ratio_term = min(1.0, inlier_ratio / max(0.01, config.conservative.min_inlier_ratio * 1.2))\n    part_term = max(0.0, min(1.0, (part_score + 0.05) / 1.05))\n    overlap_term = min(1.0, part_overlap * 2.0)\n    local_score = (\n        0.42 * inlier_term\n        + 0.24 * ratio_term\n        + 0.10 * spatial_coverage\n        + 0.16 * part_term\n        + 0.04 * overlap_term\n        + 0.04 * max(0.0, min(1.0, (visual_sim + 0.1) / 1.1))\n    )\n    if part_cells < 2 and inliers < max(10, config.balanced.min_inliers):\n        local_score *= 0.68\n    if min(a.visibility_coverage, b.visibility_coverage) < 0.12 and inliers < max(12, config.balanced.min_inliers):\n        local_score *= 0.72\n    local_score *= min(1.0, 0.65 + 0.35 * min(a.quality, b.quality))\n    return {\n        \"inliers\": inliers,\n        \"good_matches\": len(good),\n        \"inlier_ratio\": inlier_ratio,\n        \"spatial_coverage\": spatial_coverage,\n        \"local_score\": float(local_score),\n        \"visual_sim\": visual_sim,\n        \"part_score\": part_score,\n        \"part_overlap\": part_overlap,\n        \"part_cells\": part_cells,\n        \"coverage_a\": float(a.visibility_coverage),\n        \"coverage_b\": float(b.visibility_coverage),\n    }\n\n\ndef accept_edge(row: dict, rule: EdgeRule) -> bool:\n    if row[\"species\"] == \"LynxID2025\" and row[\"orientation_rule\"] == \"lynx_opposite_flank\" and not rule.allow_lynx_opposite_side:\n        return False\n    alt_vote = int(row.get(\"alt_votes\", 0))\n    min_inliers = max(4, rule.min_inliers - min(6, alt_vote * 2))\n    min_ratio = max(0.05, rule.min_inlier_ratio - min(0.04, alt_vote * 0.012))\n    min_score = max(0.10, rule.min_score - min(0.06, alt_vote * 0.018))\n    # Occlusion-aware guard: missing SAM regions are allowed, but a merge\n    # still needs either several mutually visible body zones or very strong\n    # geometric evidence. This avoids treating a tiny unmasked fragment as a\n    # full fingerprint match.\n    if int(row.get(\"part_cells\", 0)) < 2 and int(row[\"inliers\"]) < (min_inliers + 4) and alt_vote == 0:\n        return False\n    if float(row.get(\"part_overlap\", 0.0)) < 0.08 and int(row[\"inliers\"]) < int(math.ceil(min_inliers * 1.5)) and alt_vote == 0:\n        return False\n    return (\n        int(row[\"inliers\"]) >= min_inliers\n        and float(row[\"inlier_ratio\"]) >= min_ratio\n        and float(row[\"local_score\"]) >= min_score\n    )\n\n\ndef evaluate_pairs_for_species(\n    species: str,\n    species_rows: pd.DataFrame,\n    source_submissions: dict[str, pd.DataFrame],\n    args: argparse.Namespace,\n    reports_dir: Path,\n) -> tuple[list[PatternItem], pd.DataFrame, dict]:\n    config = SPECIES_CONFIGS[species]\n    detector_name, detector = create_detector(config.max_keypoints)\n    rows = species_rows.sort_values(\"image_id\").to_dict(\"records\")\n    if args.max_images_per_species:\n        rows = rows[: args.max_images_per_species]\n    if args.smoke:\n        rows = rows[: min(len(rows), 42)]\n\n    if args.save_roi_preview:\n        save_roi_preview(rows, reports_dir / f\"{VERSION}_{species}_roi_preview.jpg\", config, args.max_side)\n\n    items: list[PatternItem] = []\n    failures = 0\n    for idx, row in enumerate(rows, start=1):\n        try:\n            items.append(extract_pattern_item(row, config, args.max_side, detector_name, detector))\n        except Exception as exc:\n            failures += 1\n            fallback_vec = np.zeros(1024 + 56, dtype=np.float32)\n            n_parts = part_grid(config)[0] * part_grid(config)[1]\n            fallback_parts = np.zeros((n_parts, 96), dtype=np.float32)\n            fallback_vis = np.zeros(n_parts, dtype=bool)\n            items.append(\n                PatternItem(\n                    image_id=int(row[\"image_id\"]),\n                    row_idx=int(row.get(\"row_idx\", row[\"image_id\"])),\n                    species=species,\n                    orientation=str(row.get(\"orientation\", \"unknown\")).lower(),\n                    view_path=str(row.get(\"view_path\", row.get(\"source_path\", \"\"))),\n                    view_source=\"failed\",\n                    quality=0.0,\n                    keypoints=np.zeros((0, 2), dtype=np.float32),\n                    descriptors=None,\n                    vector=fallback_vec,\n                    fallback_vector=fallback_vec,\n                    part_vectors=fallback_parts,\n                    part_visibility=fallback_vis,\n                    fallback_part_vectors=fallback_parts,\n                    fallback_part_visibility=fallback_vis,\n                    visibility_coverage=0.0,\n                    n_keypoints=0,\n                )\n            )\n            print(f\"[warn] {species}: feature extraction failed for image_id={row.get('image_id')}: {exc}\")\n        if idx % 100 == 0:\n            print(f\"[{species}] extracted {idx}/{len(rows)}\")\n\n    current_labels = labels_for_species(source_submissions[\"current_best\"], species_rows)\n    if args.smoke or args.max_images_per_species:\n        keep_ids = {it.image_id for it in items}\n        current_labels = {k: v for k, v in current_labels.items() if k in keep_ids}\n    alt_votes = cluster_pair_votes(source_submissions, species_rows[species_rows[\"image_id\"].isin(current_labels)], config)\n    alt_votes = {k: v for k, v in alt_votes.items() if k[0] in current_labels and k[1] in current_labels}\n    pairs = shortlist_pairs(items, current_labels, alt_votes, config, args.top_k_scale, args.pair_budget_per_species)\n    item_by_id = {it.image_id: it for it in items}\n\n    score_rows = []\n    for idx, (a_id, b_id) in enumerate(sorted(pairs), start=1):\n        a = item_by_id.get(a_id)\n        b = item_by_id.get(b_id)\n        if a is None or b is None:\n            continue\n        orient_ok, orient_rule = orientation_compatible(species, a.orientation, b.orientation, allow_opposite_lynx=True)\n        scores = local_match_score(a, b, config, detector_name, config.ratio_test)\n        if species == \"LynxID2025\" and orient_rule == \"lynx_opposite_flank\":\n            scores[\"local_score\"] *= 0.55\n        same_current = current_labels.get(a_id) == current_labels.get(b_id)\n        row = {\n            \"species\": species,\n            \"image_id_a\": a_id,\n            \"image_id_b\": b_id,\n            \"orientation_a\": a.orientation,\n            \"orientation_b\": b.orientation,\n            \"orientation_rule\": orient_rule,\n            \"same_current_cluster\": bool(same_current),\n            \"alt_votes\": int(alt_votes.get((a_id, b_id), 0)),\n            \"detector\": detector_name,\n            \"kp_a\": int(a.n_keypoints),\n            \"kp_b\": int(b.n_keypoints),\n            **scores,\n        }\n        row[\"accept_conservative\"] = accept_edge(row, config.conservative)\n        row[\"accept_balanced\"] = accept_edge(row, config.balanced)\n        row[\"accept_swing\"] = accept_edge(row, config.swing)\n        score_rows.append(row)\n        if idx % 5000 == 0:\n            print(f\"[{species}] scored {idx}/{len(pairs)} pairs\")\n\n    pair_scores = pd.DataFrame(score_rows)\n    info = {\n        \"species\": species,\n        \"n_images\": len(items),\n        \"feature_failures\": failures,\n        \"detector\": detector_name,\n        \"n_pairs_scored\": int(len(pair_scores)),\n        \"mean_keypoints\": float(np.mean([it.n_keypoints for it in items])) if items else 0.0,\n        \"median_keypoints\": float(np.median([it.n_keypoints for it in items])) if items else 0.0,\n        \"mean_visibility_coverage\": float(np.mean([it.visibility_coverage for it in items])) if items else 0.0,\n        \"median_visibility_coverage\": float(np.median([it.visibility_coverage for it in items])) if items else 0.0,\n        \"resolved_sam_or_mask_views\": int(sum(it.view_source != \"original\" for it in items)),\n    }\n    if not pair_scores.empty:\n        for col in [\"accept_conservative\", \"accept_balanced\", \"accept_swing\"]:\n            info[col] = int(pair_scores[col].sum())\n        info[\"max_inliers\"] = int(pair_scores[\"inliers\"].max())\n        info[\"max_local_score\"] = float(pair_scores[\"local_score\"].max())\n        info[\"median_part_overlap\"] = float(pair_scores[\"part_overlap\"].median())\n        info[\"max_part_score\"] = float(pair_scores[\"part_score\"].max())\n    return items, pair_scores, info\n\n\ndef summarize_submission(submission: pd.DataFrame, test_rows: pd.DataFrame, current: pd.DataFrame, variant: str) -> list[dict]:\n    current_map = dict(zip(current[\"image_id\"].astype(int), current[\"cluster\"].astype(str)))\n    sub_map = dict(zip(submission[\"image_id\"].astype(int), submission[\"cluster\"].astype(str)))\n    rows = []\n    for species in SPECIES_ORDER:\n        ids = test_rows.loc[test_rows[\"species_id\"].eq(species), \"image_id\"].astype(int).tolist()\n        labels = [sub_map[i] for i in ids]\n        counts = pd.Series(labels).value_counts()\n        changed = sum(1 for i in ids if current_map.get(i) != sub_map.get(i))\n        rows.append(\n            {\n                \"variant\": variant,\n                \"species\": species,\n                \"n_images\": len(ids),\n                \"n_clusters\": int(counts.shape[0]),\n                \"max_cluster_size\": int(counts.max()) if not counts.empty else 0,\n                \"singletons\": int((counts == 1).sum()) if not counts.empty else 0,\n                \"rows_changed_vs_current\": int(changed),\n            }\n        )\n    return rows\n\n\ndef relabel_components(image_to_component: dict[int, object], species: str, variant: str) -> dict[int, str]:\n    component_order: dict[object, int] = {}\n    labels: dict[int, str] = {}\n    for image_id in sorted(image_to_component):\n        comp = image_to_component[image_id]\n        if comp not in component_order:\n            component_order[comp] = len(component_order)\n        labels[image_id] = f\"cluster_{species}_{variant}_{component_order[comp]}\"\n    return labels\n\n\ndef merge_variant_for_species(\n    species: str,\n    species_ids: list[int],\n    current_labels: dict[int, str],\n    pair_scores: pd.DataFrame,\n    profile: str,\n) -> dict[int, str]:\n    config = SPECIES_CONFIGS[species]\n    clusters = sorted({current_labels[i] for i in species_ids})\n    uf = UnionFind(clusters)\n    if pair_scores.empty:\n        return {i: current_labels[i] for i in species_ids}\n    accept_col = f\"accept_{profile}\"\n    for row in pair_scores[pair_scores[accept_col]].itertuples(index=False):\n        a = int(row.image_id_a)\n        b = int(row.image_id_b)\n        ca = current_labels.get(a)\n        cb = current_labels.get(b)\n        if ca is None or cb is None:\n            continue\n        if ca != cb:\n            uf.union(ca, cb)\n    return {i: str(uf.find(current_labels[i])) for i in species_ids}\n\n\ndef splitmerge_variant_for_species(\n    species: str,\n    species_ids: list[int],\n    current_labels: dict[int, str],\n    pair_scores: pd.DataFrame,\n) -> dict[int, str]:\n    config = SPECIES_CONFIGS[species]\n    if species not in {\"LynxID2025\", \"TexasHornedLizards\"}:\n        return merge_variant_for_species(species, species_ids, current_labels, pair_scores, \"swing\")\n\n    uf = UnionFind(species_ids)\n    groups: dict[str, list[int]] = {}\n    for image_id in species_ids:\n        groups.setdefault(current_labels[image_id], []).append(image_id)\n\n    for members in groups.values():\n        if len(members) <= config.preserve_clusters_up_to:\n            anchor = members[0]\n            for other in members[1:]:\n                uf.union(anchor, other)\n\n    if not pair_scores.empty:\n        accepted = pair_scores[pair_scores[\"accept_swing\"]].copy()\n        for row in accepted.itertuples(index=False):\n            a = int(row.image_id_a)\n            b = int(row.image_id_b)\n            if a not in current_labels or b not in current_labels:\n                continue\n            ca = current_labels[a]\n            cb = current_labels[b]\n            same_current = ca == cb\n            if same_current or bool(row.accept_balanced) or int(row.alt_votes) > 0:\n                uf.union(a, b)\n\n    comp_by_id = {i: uf.find(i) for i in species_ids}\n    return relabel_components(comp_by_id, species, \"v7_swing_splitmerge\")\n\n\ndef build_submission_variants(\n    test_rows: pd.DataFrame,\n    source_submissions: dict[str, pd.DataFrame],\n    pair_scores_by_species: dict[str, pd.DataFrame],\n) -> dict[str, pd.DataFrame]:\n    current = source_submissions[\"current_best\"].copy()\n    variants: dict[str, pd.DataFrame] = {}\n    current_base = current.copy()\n    variants[\"current_best_passthrough\"] = current_base\n\n    species_rows_by_name = {\n        species: test_rows[test_rows[\"species_id\"].eq(species)].sort_values(\"image_id\").copy()\n        for species in SPECIES_ORDER\n    }\n    current_labels_by_species = {\n        species: labels_for_species(current, rows) for species, rows in species_rows_by_name.items()\n    }\n\n    for profile in [\"conservative\", \"balanced\", \"swing\"]:\n        sub = current.copy()\n        label_updates: dict[int, str] = {}\n        for species, rows in species_rows_by_name.items():\n            ids = rows[\"image_id\"].astype(int).tolist()\n            pair_scores = pair_scores_by_species.get(species, pd.DataFrame())\n            # SeaTurtle is already the strongest local branch; keep it guarded in all but swing.\n            effective_profile = \"conservative\" if species == \"SeaTurtleID2022\" and profile != \"swing\" else profile\n            labels = merge_variant_for_species(\n                species,\n                ids,\n                current_labels_by_species[species],\n                pair_scores,\n                effective_profile,\n            )\n            label_updates.update(labels)\n        original_map = dict(zip(sub[\"image_id\"].astype(int), sub[\"cluster\"].astype(str)))\n        sub[\"cluster\"] = sub[\"image_id\"].astype(int).map(lambda i: label_updates.get(i, original_map[i]))\n        variants[f\"current_plus_{profile}_pattern_merges\"] = sub\n\n    split_sub = current.copy()\n    split_updates: dict[int, str] = {}\n    for species, rows in species_rows_by_name.items():\n        ids = rows[\"image_id\"].astype(int).tolist()\n        labels = splitmerge_variant_for_species(\n            species,\n            ids,\n            current_labels_by_species[species],\n            pair_scores_by_species.get(species, pd.DataFrame()),\n        )\n        split_updates.update(labels)\n    split_original_map = dict(zip(split_sub[\"image_id\"].astype(int), split_sub[\"cluster\"].astype(str)))\n    split_sub[\"cluster\"] = split_sub[\"image_id\"].astype(int).map(lambda i: split_updates.get(i, split_original_map[i]))\n    variants[\"swing_split_large_lynx_texas\"] = split_sub\n    return variants\n\n\ndef update_experiment_log(output_root: Path, summary: dict, candidate_report: pd.DataFrame, best_path: Path) -> None:\n    log_path = Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\current_wildfusion_graph_v20260423\\experiment_log.json\")\n    if not log_path.exists():\n        return\n    try:\n        log = json.loads(log_path.read_text(encoding=\"utf-8\"))\n    except Exception:\n        return\n    entry = {\n        \"run_id\": VERSION,\n        \"date\": \"2026-04-26\",\n        \"status\": \"implemented_smoke_tested_ready_for_kaggle\",\n        \"goal\": \"Final-swing species-specific local fingerprint ensemble over current 0.29758 hybrid.\",\n        \"output_root\": str(output_root),\n        \"recommended_first_submission\": str(best_path),\n        \"summary\": summary,\n        \"candidate_report_preview\": candidate_report.to_dict(\"records\")[:20],\n    }\n    if isinstance(log, list):\n        log = [run for run in log if not (isinstance(run, dict) and run.get(\"run_id\") == VERSION)]\n        log.append(entry)\n    elif isinstance(log, dict):\n        runs = log.setdefault(\"runs\", [])\n        log[\"runs\"] = [run for run in runs if not (isinstance(run, dict) and run.get(\"run_id\") == VERSION)]\n        log[\"runs\"].append(entry)\n    else:\n        return\n    log_path.write_text(json.dumps(log, indent=2), encoding=\"utf-8\")\n\n\ndef main() -> None:\n    random.seed(SEED)\n    np.random.seed(SEED)\n    args = parse_args()\n    if args.smoke:\n        args.max_images_per_species = args.max_images_per_species or 42\n        args.pair_budget_per_species = min(args.pair_budget_per_species, 2500)\n        args.top_k_scale = min(args.top_k_scale, 0.55)\n\n    data_root = find_data_root(args.data_root)\n    sam_manifest = find_sam_manifest(args.sam_manifest)\n    output_root = Path(args.output_root) if args.output_root else Path.cwd() / f\"animalclef_{VERSION}\"\n    reports_dir = output_root / \"reports\"\n    submissions_dir = output_root / \"submissions\"\n    reports_dir.mkdir(parents=True, exist_ok=True)\n    submissions_dir.mkdir(parents=True, exist_ok=True)\n\n    test_rows, manifest_info = prepare_test_metadata(data_root, sam_manifest)\n    selected_species = [s for s in args.species if s in SPECIES_CONFIGS]\n    test_rows = test_rows[test_rows[\"species_id\"].isin(SPECIES_ORDER)].copy()\n    source_paths = find_submission_sources(args, data_root)\n    source_submissions = {name: load_submission(path) for name, path in source_paths.items()}\n\n    print(f\"VERSION={VERSION}\")\n    print(f\"data_root={data_root}\")\n    print(f\"sam_manifest={sam_manifest}\")\n    print(f\"output_root={output_root}\")\n    print(\"submission sources:\")\n    for name, path in source_paths.items():\n        print(f\"  {name}: {path}\")\n\n    pair_scores_by_species: dict[str, pd.DataFrame] = {}\n    item_infos: list[dict] = []\n    for species in selected_species:\n        species_rows = test_rows[test_rows[\"species_id\"].eq(species)].copy()\n        items, pair_scores, info = evaluate_pairs_for_species(species, species_rows, source_submissions, args, reports_dir)\n        pair_scores_by_species[species] = pair_scores\n        item_infos.append(info)\n        item_manifest = pd.DataFrame(\n            [\n                {\n                    \"species\": it.species,\n                    \"image_id\": it.image_id,\n                    \"row_idx\": it.row_idx,\n                    \"orientation\": it.orientation,\n                    \"view_path\": it.view_path,\n                    \"view_source\": it.view_source,\n                    \"quality\": it.quality,\n                    \"visibility_coverage\": it.visibility_coverage,\n                    \"visible_parts\": int(it.part_visibility.sum()),\n                    \"fallback_visible_parts\": int(it.fallback_part_visibility.sum()),\n                    \"n_keypoints\": it.n_keypoints,\n                }\n                for it in items\n            ]\n        )\n        item_manifest.to_csv(reports_dir / f\"{VERSION}_{species}_item_manifest.csv\", index=False)\n        pair_scores.to_csv(reports_dir / f\"{VERSION}_{species}_pair_scores.csv\", index=False)\n\n    nonempty_pair_scores = [df for df in pair_scores_by_species.values() if not df.empty]\n    all_pair_scores = pd.concat(nonempty_pair_scores, ignore_index=True) if nonempty_pair_scores else pd.DataFrame()\n    all_pair_scores.to_csv(reports_dir / f\"{VERSION}_all_pair_scores.csv\", index=False)\n\n    variants = build_submission_variants(test_rows, source_submissions, pair_scores_by_species)\n    current = source_submissions[\"current_best\"]\n    report_rows = []\n    written_paths = {}\n    for variant, sub in variants.items():\n        out_path = submissions_dir / f\"submission_{VERSION}_{variant}.csv\"\n        sub.to_csv(out_path, index=False)\n        written_paths[variant] = str(out_path)\n        report_rows.extend(summarize_submission(sub, test_rows, current, variant))\n\n    candidate_report = pd.DataFrame(report_rows)\n    candidate_report.to_csv(reports_dir / f\"{VERSION}_candidate_report.csv\", index=False)\n\n    recommended = submissions_dir / f\"submission_{VERSION}_current_plus_conservative_pattern_merges.csv\"\n    summary = {\n        \"version\": VERSION,\n        \"data_root\": str(data_root),\n        \"sam_manifest\": str(sam_manifest) if sam_manifest else None,\n        \"manifest_info\": manifest_info,\n        \"selected_species\": selected_species,\n        \"source_paths\": {k: str(v) for k, v in source_paths.items()},\n        \"item_infos\": item_infos,\n        \"written_submissions\": written_paths,\n        \"recommended_first_submission\": str(recommended),\n        \"notes\": [\n            \"Conservative/balanced/swing merge variants preserve the current 0.29758 hybrid and only merge clusters with local pattern evidence.\",\n            \"Occlusion-aware visibility grids ignore SAM-removed or hand-corrupted body zones instead of treating missing parts as mismatches.\",\n            \"swing_split_large_lynx_texas is the high-risk/high-upside file; it can repair oversized Lynx/Texas clusters but may over-split.\",\n            \"SeaTurtle uses guarded rules because p06 is already very strong locally.\",\n        ],\n    }\n    (reports_dir / f\"{VERSION}_summary.json\").write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n    update_experiment_log(output_root, summary, candidate_report, recommended)\n\n    print(\"candidate report:\")\n    print(candidate_report.to_string(index=False))\n    print(f\"wrote {recommended}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
Path("texas_astrodot_2025reuse_v20260426.py").write_text("from __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nimport math\nimport random\nimport sys\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw, ImageFile\n\n\nImageFile.LOAD_TRUNCATED_IMAGES = True\n\nTHIS_DIR = Path(__file__).resolve().parent\nif str(THIS_DIR) not in sys.path:\n    sys.path.insert(0, str(THIS_DIR))\n\nimport species_fingerprint_final_swing_v20260426 as core\n\n\nVERSION = \"texas_astrodot_2025reuse_v20260426\"\nSEED = 20260426\nTEXAS = \"TexasHornedLizards\"\nREUSED_SPECIES = [\"LynxID2025\", \"SalamanderID2025\", \"SeaTurtleID2022\"]\nBACKGROUND = np.array([238, 238, 232], dtype=np.uint8)\n\n\n@dataclass\nclass TexasDotItem:\n    image_id: int\n    current_cluster: str\n    source_path: str\n    view_path: str\n    view_source: str\n    belly_rgb: np.ndarray\n    belly_mask: np.ndarray\n    dot_heat: np.ndarray\n    dot_points: np.ndarray\n    vector: np.ndarray\n    quality: float\n    debug: dict\n\n\nclass UnionFind:\n    def __init__(self, values: Iterable[int]):\n        self.parent = {int(v): int(v) for v in values}\n        self.size = {int(v): 1 for v in values}\n\n    def find(self, value: int) -> int:\n        value = int(value)\n        if value not in self.parent:\n            self.parent[value] = value\n            self.size[value] = 1\n            return value\n        root = value\n        while self.parent[root] != root:\n            root = self.parent[root]\n        while self.parent[value] != value:\n            nxt = self.parent[value]\n            self.parent[value] = root\n            value = nxt\n        return root\n\n    def union(self, a: int, b: int) -> int:\n        ra = self.find(a)\n        rb = self.find(b)\n        if ra == rb:\n            return ra\n        if self.size[ra] < self.size[rb]:\n            ra, rb = rb, ra\n        self.parent[rb] = ra\n        self.size[ra] += self.size[rb]\n        return ra\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=(\n            \"Texas astro-dot stacking branch plus 2025-winner-style train \"\n            \"local-match validation for reused AnimalCLEF species. This script \"\n            \"writes upload-ready CSVs only; it never submits to Kaggle.\"\n        )\n    )\n    parser.add_argument(\"--data-root\", type=str, default=None)\n    parser.add_argument(\"--sam-manifest\", type=str, default=None)\n    parser.add_argument(\"--current-best-submission\", type=str, default=None)\n    parser.add_argument(\"--output-root\", type=str, default=None)\n    parser.add_argument(\"--max-side\", type=int, default=760)\n    parser.add_argument(\"--texas-canvas-w\", type=int, default=224)\n    parser.add_argument(\"--texas-canvas-h\", type=int, default=320)\n    parser.add_argument(\"--texas-pair-budget\", type=int, default=12000, help=\"0 means all Texas pairs.\")\n    parser.add_argument(\"--reused-train-pair-budget\", type=int, default=1800)\n    parser.add_argument(\"--reused-max-train-images\", type=int, default=650)\n    parser.add_argument(\"--reused-max-per-identity\", type=int, default=8)\n    parser.add_argument(\"--skip-reused-validation\", action=\"store_true\")\n    parser.add_argument(\"--save-visualizations\", action=\"store_true\")\n    parser.add_argument(\"--visual-limit\", type=int, default=18)\n    parser.add_argument(\"--smoke\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef find_data_root(user_value: str | None) -> Path:\n    candidates: list[Path] = []\n    if user_value:\n        candidates.append(Path(user_value))\n    candidates.extend(\n        [\n            Path(\"/kaggle/input/animal-clef-2026\"),\n            Path(\"/kaggle/input/competitions/animal-clef-2026\"),\n            Path.cwd() / \"animal-clef-2026\",\n            Path.cwd().parent / \"animal-clef-2026\",\n            Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\animal-clef-2026\"),\n        ]\n    )\n    for root in candidates:\n        if (root / \"metadata.csv\").exists() and (root / \"sample_submission.csv\").exists():\n            return root.resolve()\n    raise FileNotFoundError(\"Could not locate AnimalCLEF2026 data root.\")\n\n\ndef find_sam_manifest(user_value: str | None) -> Path | None:\n    if user_value:\n        p = Path(user_value)\n        if p.exists():\n            return p.resolve()\n    direct = [\n        Path(\"/kaggle/working/animalclef_sam3_views_cache/reports/view_manifest_sam3_all_species.csv\"),\n        Path(\"/kaggle/input/animalclef2026-export-sam3-views-all-species-v2026/animalclef_sam3_views_cache/reports/view_manifest_sam3_all_species.csv\"),\n        Path(\"/kaggle/input/animalclef2026-export-sam3-views-all-species-v2026/reports/view_manifest_sam3_all_species.csv\"),\n        Path.cwd() / \"animalclef_sam3_views_cache\" / \"reports\" / \"view_manifest_sam3_all_species.csv\",\n        Path.cwd().parent / \"animalclef_sam3_views_cache\" / \"reports\" / \"view_manifest_sam3_all_species.csv\",\n    ]\n    for p in direct:\n        if p.exists():\n            return p.resolve()\n    for base in [Path(\"/kaggle/input\"), Path.cwd(), Path.cwd().parent]:\n        if not base.exists():\n            continue\n        try:\n            matches = list(base.rglob(\"view_manifest_sam3_all_species.csv\"))\n        except Exception:\n            matches = []\n        if matches:\n            matches.sort(key=lambda x: len(str(x)))\n            return matches[0].resolve()\n    return None\n\n\ndef export_root_from_manifest(manifest_path: Path) -> Path:\n    if manifest_path.parent.name == \"reports\":\n        return manifest_path.parent.parent\n    return manifest_path.parent\n\n\ndef remap_export_path(path_value: object, export_root: Path | None) -> Path | None:\n    if path_value is None or pd.isna(path_value):\n        return None\n    s = str(path_value).strip()\n    if not s:\n        return None\n    p = Path(s)\n    if p.exists():\n        return p.resolve()\n    if export_root is None:\n        return None\n    normalized = s.replace(\"\\\\\", \"/\")\n    markers = [\n        \"animalclef_sam3_views_cache/\",\n        \"views/\",\n        \"mask_loose_square/\",\n        \"mask_full_square/\",\n    ]\n    for marker in markers:\n        if marker not in normalized:\n            continue\n        rel = normalized.split(marker, 1)[1]\n        if marker != \"animalclef_sam3_views_cache/\":\n            rel = marker + rel\n        candidate = export_root / Path(rel)\n        if candidate.exists():\n            return candidate.resolve()\n    return None\n\n\ndef prepare_metadata(data_root: Path, sam_manifest: Path | None) -> tuple[pd.DataFrame, dict]:\n    metadata = pd.read_csv(data_root / \"metadata.csv\").reset_index(drop=True)\n    if \"row_idx\" not in metadata.columns:\n        metadata[\"row_idx\"] = np.arange(len(metadata), dtype=np.int64)\n    if \"species_id\" not in metadata.columns:\n        metadata[\"species_id\"] = metadata[\"dataset\"].astype(str)\n    if \"split\" not in metadata.columns:\n        metadata[\"split\"] = np.where(metadata[\"path\"].str.contains(\"/test/\"), \"test\", \"train\")\n    if \"orientation\" not in metadata.columns:\n        metadata[\"orientation\"] = \"unknown\"\n    if \"identity\" not in metadata.columns:\n        metadata[\"identity\"] = \"\"\n    metadata[\"source_path\"] = metadata[\"path\"].map(lambda p: str(data_root / str(p)))\n    metadata[\"view_path\"] = metadata[\"source_path\"].astype(str)\n    metadata[\"view_source\"] = \"original_fallback\"\n    metadata[\"sam_view_path\"] = \"\"\n    metadata[\"mask_path\"] = \"\"\n    metadata[\"mask_ok\"] = False\n\n    info = {\n        \"manifest_path\": str(sam_manifest) if sam_manifest else None,\n        \"manifest_rows\": 0,\n        \"resolved_sam_views\": 0,\n        \"resolved_masks\": 0,\n    }\n    if sam_manifest is None:\n        return metadata, info\n\n    manifest = pd.read_csv(sam_manifest)\n    info[\"manifest_rows\"] = int(len(manifest))\n    export_root = export_root_from_manifest(sam_manifest)\n    merge_key = \"row_idx\" if \"row_idx\" in manifest.columns else \"image_id\" if \"image_id\" in manifest.columns else None\n    if merge_key is None:\n        return metadata, info\n    merged = metadata.merge(manifest, on=merge_key, how=\"left\", suffixes=(\"\", \"_sam\"))\n\n    view_paths: list[str] = []\n    sam_view_paths: list[str] = []\n    mask_paths: list[str] = []\n    view_sources: list[str] = []\n    mask_ok: list[bool] = []\n    sam_count = 0\n    mask_count = 0\n    for row in merged.to_dict(\"records\"):\n        resolved_sam_view = None\n        for col in [\"loose_path\", \"mask_loose_path\", \"mask_full_path\", \"view_path\"]:\n            if col in row:\n                resolved_sam_view = remap_export_path(row.get(col), export_root)\n                if resolved_sam_view is not None:\n                    break\n        resolved_mask = None\n        for col in [\"mask_path\", \"binary_mask_path\"]:\n            if col in row:\n                resolved_mask = remap_export_path(row.get(col), export_root)\n                if resolved_mask is not None:\n                    break\n        if resolved_sam_view is not None:\n            view_paths.append(str(resolved_sam_view))\n            sam_view_paths.append(str(resolved_sam_view))\n            view_sources.append(\"sam_clean_primary\")\n            sam_count += 1\n        else:\n            view_paths.append(str(row[\"source_path\"]))\n            sam_view_paths.append(\"\")\n            view_sources.append(\"original_fallback\")\n        if resolved_mask is not None:\n            mask_paths.append(str(resolved_mask))\n            mask_ok.append(True)\n            mask_count += 1\n        else:\n            mask_paths.append(\"\")\n            mask_ok.append(False)\n    merged[\"view_path\"] = view_paths\n    merged[\"sam_view_path\"] = sam_view_paths\n    merged[\"view_source\"] = view_sources\n    merged[\"mask_path\"] = mask_paths\n    merged[\"mask_ok\"] = mask_ok\n    info[\"resolved_sam_views\"] = int(sam_count)\n    info[\"resolved_masks\"] = int(mask_count)\n    return merged, info\n\n\ndef find_current_best(user_value: str | None, data_root: Path) -> Path:\n    if user_value:\n        p = Path(user_value)\n        if p.exists():\n            return p.resolve()\n    filename = core.CURRENT_BEST_FILENAME\n    roots = [\n        Path(\"/kaggle/input\"),\n        Path(\"/kaggle/working\"),\n        Path.cwd(),\n        Path.cwd().parent,\n        data_root.parent,\n        Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\current_wildfusion_graph_v20260423\"),\n        Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\kaggle_upload_atlas_inputs_v20260426\"),\n    ]\n    found = core.find_file_everywhere(filename, roots)\n    if found is None:\n        raise FileNotFoundError(f\"Could not find {filename}.\")\n    return found\n\n\ndef normalize(vec: np.ndarray) -> np.ndarray:\n    vec = vec.astype(np.float32, copy=False)\n    norm = float(np.linalg.norm(vec))\n    if norm > 0:\n        vec = vec / norm\n    return vec.astype(np.float32, copy=False)\n\n\ndef clean_mask(mask: np.ndarray, shape: tuple[int, int] | None = None) -> np.ndarray:\n    m = np.where(mask > 0, 255, 0).astype(np.uint8)\n    if shape is not None and m.shape[:2] != shape:\n        m = cv2.resize(m, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)\n        m = np.where(m > 0, 255, 0).astype(np.uint8)\n    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))\n    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))\n    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k1)\n    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k2)\n    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, 8)\n    if n > 1:\n        areas = stats[1:, cv2.CC_STAT_AREA]\n        if len(areas):\n            biggest = 1 + int(np.argmax(areas))\n            m = np.where(labels == biggest, 255, 0).astype(np.uint8)\n    if float(m.mean() / 255.0) < 0.01:\n        m = np.full(m.shape[:2], 255, dtype=np.uint8)\n    return m\n\n\ndef read_rgb_mask(row: dict, max_side: int) -> tuple[np.ndarray, np.ndarray, float]:\n    rgb, mask = core.read_rgb_with_optional_mask(row.get(\"view_path\", row.get(\"source_path\")), row.get(\"mask_path\"), max_side)\n    quality = 1.0\n    if mask is None:\n        mask = core.estimate_foreground_mask(rgb)\n        quality = 0.86\n    mask = clean_mask(mask, rgb.shape[:2])\n    coverage = float(mask.mean() / 255.0)\n    if coverage < 0.015 or coverage > 0.98:\n        mask = core.estimate_foreground_mask(rgb)\n        mask = clean_mask(mask, rgb.shape[:2])\n        quality *= 0.78\n    return rgb, mask, quality\n\n\ndef crop_to_mask(rgb: np.ndarray, mask: np.ndarray, pad: float = 0.08) -> tuple[np.ndarray, np.ndarray]:\n    x1, y1, x2, y2 = core.bbox_from_mask(mask, pad)\n    return rgb[y1:y2, x1:x2].copy(), mask[y1:y2, x1:x2].copy()\n\n\ndef align_vertical(rgb: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray, dict]:\n    crop_rgb, crop_mask = crop_to_mask(rgb, mask, 0.08)\n    angle = core.pca_angle_degrees(crop_mask)\n    rotate_angle = 90.0 - angle\n    if abs(rotate_angle) > 1.5:\n        crop_rgb, crop_mask = core.rotate_bound(crop_rgb, crop_mask, rotate_angle)\n        crop_rgb, crop_mask = crop_to_mask(crop_rgb, crop_mask, 0.04)\n    h, w = crop_mask.shape[:2]\n    widths = []\n    for yf in [0.18, 0.30, 0.70, 0.84]:\n        y = int(np.clip(round(h * yf), 0, h - 1))\n        xs = np.where(crop_mask[y] > 0)[0]\n        widths.append(float(xs.max() - xs.min() + 1) if len(xs) else 0.0)\n    top_width = max(widths[:2])\n    bottom_width = max(widths[2:])\n    flipped = False\n    # Most Texas belly photos have the head/shoulder end above the tail end.\n    # If the bottom bands are clearly wider, canonicalize by vertical flip.\n    if bottom_width > top_width * 1.18:\n        crop_rgb = crop_rgb[::-1, :, :].copy()\n        crop_mask = crop_mask[::-1, :].copy()\n        flipped = True\n    return crop_rgb, crop_mask, {\n        \"pca_angle\": float(angle),\n        \"rotate_angle\": float(rotate_angle),\n        \"top_width\": float(top_width),\n        \"bottom_width\": float(bottom_width),\n        \"flipped_vertical\": bool(flipped),\n    }\n\n\ndef clahe_u8(channel: np.ndarray) -> np.ndarray:\n    channel = np.clip(channel, 0, 255).astype(np.uint8)\n    return cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(channel)\n\n\ndef texas_dot_heat(belly_rgb: np.ndarray, belly_mask: np.ndarray) -> np.ndarray:\n    lab = cv2.cvtColor(belly_rgb, cv2.COLOR_RGB2LAB)\n    l_eq = clahe_u8(lab[:, :, 0])\n    dark = (255.0 - l_eq.astype(np.float32)) / 255.0\n    blackhat = cv2.morphologyEx(l_eq, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13)))\n    spot = 0.58 * (blackhat.astype(np.float32) / 255.0) + 0.42 * dark\n    valid = belly_mask > 0\n    if valid.sum() > 20:\n        vals = spot[valid]\n        lo = float(np.percentile(vals, 12))\n        hi = float(np.percentile(vals, 98))\n        spot = (spot - lo) / max(1e-6, hi - lo)\n    spot = np.clip(spot, 0, 1)\n    spot[~valid] = 0.0\n    spot = cv2.GaussianBlur(spot, (3, 3), 0)\n    return spot.astype(np.float32)\n\n\ndef texas_belly_template(row: dict, current_cluster: str, args: argparse.Namespace) -> TexasDotItem:\n    rgb, mask, quality = read_rgb_mask(row, args.max_side)\n    aligned_rgb, aligned_mask, debug = align_vertical(rgb, mask)\n    w = int(args.texas_canvas_w)\n    h = int(args.texas_canvas_h)\n    belly_rgb = cv2.resize(aligned_rgb, (w, h), interpolation=cv2.INTER_AREA)\n    belly_mask = cv2.resize(aligned_mask, (w, h), interpolation=cv2.INTER_NEAREST)\n    belly_mask = clean_mask(belly_mask)\n    ellipse = np.zeros((h, w), dtype=np.uint8)\n    cv2.ellipse(\n        ellipse,\n        (w // 2, int(h * 0.49)),\n        (int(w * 0.38), int(h * 0.35)),\n        0,\n        0,\n        360,\n        255,\n        -1,\n    )\n    belly_mask = cv2.bitwise_and(belly_mask, ellipse)\n    if float(belly_mask.mean() / 255.0) < 0.035:\n        belly_mask = ellipse\n        quality *= 0.72\n    heat = texas_dot_heat(belly_rgb, belly_mask)\n    points = dot_points_from_heat(heat, belly_mask)\n    small = cv2.resize(heat, (32, 48), interpolation=cv2.INTER_AREA).reshape(-1)\n    mask_small = cv2.resize((belly_mask > 0).astype(np.float32), (32, 48), interpolation=cv2.INTER_AREA).reshape(-1)\n    vector = normalize(np.concatenate([small * mask_small, mask_small * 0.18]).astype(np.float32))\n    quality *= min(1.0, 0.72 + 0.28 * min(1.0, len(points) / 35.0))\n    return TexasDotItem(\n        image_id=int(row[\"image_id\"]),\n        current_cluster=str(current_cluster),\n        source_path=str(row.get(\"source_path\", \"\")),\n        view_path=str(row.get(\"view_path\", row.get(\"source_path\", \"\"))),\n        view_source=str(row.get(\"view_source\", \"\")),\n        belly_rgb=belly_rgb,\n        belly_mask=belly_mask,\n        dot_heat=heat,\n        dot_points=points,\n        vector=vector,\n        quality=float(quality),\n        debug=debug,\n    )\n\n\ndef dot_points_from_heat(heat: np.ndarray, mask: np.ndarray) -> np.ndarray:\n    valid = mask > 0\n    if int(valid.sum()) < 40:\n        return np.zeros((0, 4), dtype=np.float32)\n    vals = heat[valid]\n    thr = max(float(np.percentile(vals, 86)), float(vals.mean() + 0.60 * vals.std()))\n    binary = np.zeros(heat.shape, dtype=np.uint8)\n    binary[(heat >= thr) & valid] = 255\n    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))\n    n, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, 8)\n    h, w = heat.shape[:2]\n    total = float(h * w)\n    pts: list[list[float]] = []\n    min_area = max(2.0, total * 0.00005)\n    max_area = total * 0.012\n    for idx in range(1, n):\n        area = float(stats[idx, cv2.CC_STAT_AREA])\n        if area < min_area or area > max_area:\n            continue\n        bw = max(1.0, float(stats[idx, cv2.CC_STAT_WIDTH]))\n        bh = max(1.0, float(stats[idx, cv2.CC_STAT_HEIGHT]))\n        if max(bw / bh, bh / bw) > 5.5:\n            continue\n        cx, cy = centroids[idx]\n        strength = float(heat[labels == idx].mean())\n        pts.append([float(cx) / max(1, w - 1), float(cy) / max(1, h - 1), area / total, strength])\n    if not pts:\n        return np.zeros((0, 4), dtype=np.float32)\n    pts_arr = np.asarray(pts, dtype=np.float32)\n    order = np.argsort(-pts_arr[:, 3])\n    return pts_arr[order[:260]]\n\n\ndef shifted_slices(shape: tuple[int, int], dx: int, dy: int):\n    h, w = shape\n    xa1 = max(0, dx)\n    xb1 = max(0, -dx)\n    ya1 = max(0, dy)\n    yb1 = max(0, -dy)\n    ww = w - abs(dx)\n    hh = h - abs(dy)\n    if ww <= 8 or hh <= 8:\n        return None\n    return (slice(ya1, ya1 + hh), slice(xa1, xa1 + ww)), (slice(yb1, yb1 + hh), slice(xb1, xb1 + ww))\n\n\ndef dot_map_sharpness(heat: np.ndarray, mask: np.ndarray) -> float:\n    valid = mask > 0\n    if int(valid.sum()) < 20:\n        return 0.0\n    vals = heat[valid].astype(np.float32)\n    p95 = float(np.percentile(vals, 95))\n    p50 = float(np.percentile(vals, 50))\n    lap = cv2.Laplacian(heat, cv2.CV_32F, ksize=3)\n    lap_energy = float(np.mean(np.abs(lap[valid])))\n    return float(max(0.0, p95 - p50) + 0.35 * lap_energy)\n\n\ndef masked_corr_and_stack(\n    a_heat: np.ndarray,\n    a_mask: np.ndarray,\n    b_heat: np.ndarray,\n    b_mask: np.ndarray,\n    dx: int,\n    dy: int,\n) -> tuple[float, float, float]:\n    slices = shifted_slices(a_mask.shape[:2], dx, dy)\n    if slices is None:\n        return 0.0, 0.0, 0.0\n    sa, sb = slices\n    ma = a_mask[sa] > 0\n    mb = b_mask[sb] > 0\n    overlap_mask = ma & mb\n    overlap = float(overlap_mask.mean()) if overlap_mask.size else 0.0\n    if overlap < 0.04 or int(overlap_mask.sum()) < 40:\n        return 0.0, overlap, 0.0\n    va = a_heat[sa][overlap_mask].astype(np.float32)\n    vb = b_heat[sb][overlap_mask].astype(np.float32)\n    am = va - float(va.mean())\n    bm = vb - float(vb.mean())\n    denom = float(np.linalg.norm(am) * np.linalg.norm(bm))\n    corr = float(np.dot(am, bm) / denom) if denom > 1e-6 else 0.0\n    corr01 = float(np.clip((corr + 1.0) * 0.5, 0.0, 1.0))\n    stack = np.zeros_like(a_heat, dtype=np.float32)\n    stack_mask = np.zeros_like(a_mask, dtype=np.uint8)\n    stack[sa] = 0.5 * (a_heat[sa] + b_heat[sb])\n    stack_mask[sa] = np.where(overlap_mask, 255, 0).astype(np.uint8)\n    sharp_stack = dot_map_sharpness(stack, stack_mask)\n    sharp_a = dot_map_sharpness(a_heat[sa], np.where(overlap_mask, 255, 0).astype(np.uint8))\n    sharp_b = dot_map_sharpness(b_heat[sb], np.where(overlap_mask, 255, 0).astype(np.uint8))\n    baseline = 0.5 * (sharp_a + sharp_b)\n    # Same individuals should retain or improve peakiness after stacking;\n    # different dot fields smear and reduce the normalized sharpness.\n    stack_gain = float(np.clip(sharp_stack / max(1e-6, baseline), 0.0, 1.35) / 1.35)\n    return corr01, overlap, stack_gain\n\n\ndef chamfer_dot_score(points_a: np.ndarray, points_b: np.ndarray, dx_norm: float, dy_norm: float) -> float:\n    if len(points_a) < 5 or len(points_b) < 5:\n        return 0.0\n    a = points_a[:, :2].astype(np.float32)\n    b = points_b[:, :2].astype(np.float32).copy()\n    b[:, 0] += dx_norm\n    b[:, 1] += dy_norm\n    diff = a[:, None, :] - b[None, :, :]\n    dist = np.sqrt(np.maximum(0.0, (diff * diff).sum(axis=2)))\n    da = dist.min(axis=1)\n    db = dist.min(axis=0)\n    ka = np.argsort(da)[: min(len(da), 120)]\n    kb = np.argsort(db)[: min(len(db), 120)]\n    mean_d = 0.5 * (float(da[ka].mean()) + float(db[kb].mean()))\n    return float(np.exp(-mean_d / 0.050))\n\n\ndef phase_shift(a: np.ndarray, b: np.ndarray, mask: np.ndarray) -> tuple[int, int]:\n    try:\n        aw = (a * mask).astype(np.float32)\n        bw = (b * mask).astype(np.float32)\n        shift, _ = cv2.phaseCorrelate(aw, bw)\n        dx = int(np.clip(round(shift[0]), -18, 18))\n        dy = int(np.clip(round(shift[1]), -18, 18))\n        return dx, dy\n    except Exception:\n        return 0, 0\n\n\ndef transform_heat_mask_points(item: TexasDotItem, transform: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\n    heat = item.dot_heat\n    mask = item.belly_mask\n    pts = item.dot_points.copy()\n    if transform in {\"flip_x\", \"flip_xy\"}:\n        heat = heat[:, ::-1].copy()\n        mask = mask[:, ::-1].copy()\n        if len(pts):\n            pts[:, 0] = 1.0 - pts[:, 0]\n    if transform in {\"flip_y\", \"flip_xy\"}:\n        heat = heat[::-1, :].copy()\n        mask = mask[::-1, :].copy()\n        if len(pts):\n            pts[:, 1] = 1.0 - pts[:, 1]\n    return heat, mask, pts\n\n\ndef texas_pair_score(a: TexasDotItem, b: TexasDotItem) -> dict:\n    desc_cos = float(np.dot(a.vector, b.vector))\n    h, w = a.dot_heat.shape[:2]\n    best = {\n        \"score\": 0.0,\n        \"corr\": 0.0,\n        \"overlap\": 0.0,\n        \"stack_gain\": 0.0,\n        \"point_score\": 0.0,\n        \"descriptor_cosine\": desc_cos,\n        \"dx\": 0,\n        \"dy\": 0,\n        \"transform\": \"identity\",\n    }\n    transforms = [\"identity\", \"flip_y\"]\n    common_mask = np.where((a.belly_mask > 0), 1.0, 0.0).astype(np.float32)\n    for transform in transforms:\n        b_heat, b_mask, b_pts = transform_heat_mask_points(b, transform)\n        base_dx, base_dy = phase_shift(a.dot_heat, b_heat, common_mask)\n        candidates = {(0, 0), (base_dx, base_dy)}\n        for dx0, dy0 in [(base_dx, base_dy), (0, 0)]:\n            for ddx in (-8, 0, 8):\n                for ddy in (-8, 0, 8):\n                    candidates.add((int(np.clip(dx0 + ddx, -24, 24)), int(np.clip(dy0 + ddy, -24, 24))))\n        for dx, dy in candidates:\n            corr, overlap, stack_gain = masked_corr_and_stack(a.dot_heat, a.belly_mask, b_heat, b_mask, dx, dy)\n            if overlap < 0.045:\n                continue\n            point_score = chamfer_dot_score(a.dot_points, b_pts, dx / max(1, w - 1), dy / max(1, h - 1))\n            fused = (\n                0.34 * corr\n                + 0.31 * point_score\n                + 0.23 * stack_gain\n                + 0.12 * max(0.0, desc_cos)\n            )\n            if transform != \"identity\":\n                fused *= 0.92\n            fused *= min(a.quality, b.quality)\n            if fused > best[\"score\"]:\n                best = {\n                    \"score\": float(fused),\n                    \"corr\": float(corr),\n                    \"overlap\": float(overlap),\n                    \"stack_gain\": float(stack_gain),\n                    \"point_score\": float(point_score),\n                    \"descriptor_cosine\": desc_cos,\n                    \"dx\": int(dx),\n                    \"dy\": int(dy),\n                    \"transform\": transform,\n                }\n    best[\"points_a\"] = int(len(a.dot_points))\n    best[\"points_b\"] = int(len(b.dot_points))\n    best[\"quality_a\"] = float(a.quality)\n    best[\"quality_b\"] = float(b.quality)\n    best[\"same_current_cluster\"] = bool(a.current_cluster == b.current_cluster)\n    return best\n\n\ndef score_all_texas_pairs(items: list[TexasDotItem], pair_budget: int = 0) -> pd.DataFrame:\n    vectors = np.stack([it.vector for it in items]).astype(np.float32)\n    sim = vectors @ vectors.T\n    ids = [it.image_id for it in items]\n    by_id = {it.image_id: it for it in items}\n    pairs = [(ids[i], ids[j]) for i in range(len(ids)) for j in range(i + 1, len(ids))]\n    if pair_budget and len(pairs) > pair_budget:\n        id_to_idx = {image_id: idx for idx, image_id in enumerate(ids)}\n        pairs.sort(key=lambda p: float(sim[id_to_idx[p[0]], id_to_idx[p[1]]]), reverse=True)\n        current_pairs = [p for p in pairs if by_id[p[0]].current_cluster == by_id[p[1]].current_cluster]\n        selected = current_pairs + [p for p in pairs if by_id[p[0]].current_cluster != by_id[p[1]].current_cluster]\n        pairs = selected[:pair_budget]\n    rows = []\n    for idx, (a_id, b_id) in enumerate(pairs, start=1):\n        a = by_id[int(a_id)]\n        b = by_id[int(b_id)]\n        score = texas_pair_score(a, b)\n        rows.append(\n            {\n                \"species\": TEXAS,\n                \"image_id_a\": int(a_id),\n                \"image_id_b\": int(b_id),\n                \"current_cluster_a\": a.current_cluster,\n                \"current_cluster_b\": b.current_cluster,\n                **score,\n            }\n        )\n        if idx % 5000 == 0:\n            print(f\"[Texas astro-dot] scored {idx}/{len(pairs)} pairs\")\n    return pd.DataFrame(rows)\n\n\ndef relabel_components(ids: list[int], uf: UnionFind, variant: str) -> dict[int, str]:\n    comp_order: dict[int, int] = {}\n    labels: dict[int, str] = {}\n    for image_id in sorted(ids):\n        comp = uf.find(image_id)\n        if comp not in comp_order:\n            comp_order[comp] = len(comp_order)\n        labels[image_id] = f\"cluster_TexasHornedLizards_astrodot_{variant}_{comp_order[comp]}\"\n    return labels\n\n\ndef texas_variant_labels(items: list[TexasDotItem], pair_scores: pd.DataFrame, variant: str) -> dict[int, str]:\n    ids = [it.image_id for it in items]\n    by_cluster: dict[str, list[int]] = {}\n    current_by_id = {it.image_id: it.current_cluster for it in items}\n    for it in items:\n        by_cluster.setdefault(it.current_cluster, []).append(it.image_id)\n\n    # Thresholds are deliberately strict. Texas has no train labels, so this\n    # branch should produce surgical candidates, not a free-running clusterer.\n    if variant == \"split_strict\":\n        keep_thr, merge_thr, merge_rank = 0.430, 9.9, 0\n    elif variant == \"merge_ultra\":\n        keep_thr, merge_thr, merge_rank = 0.0, 0.650, 2\n    elif variant == \"splitmerge_guarded\":\n        keep_thr, merge_thr, merge_rank = 0.405, 0.620, 2\n    else:\n        keep_thr, merge_thr, merge_rank = 0.385, 0.590, 3\n\n    uf = UnionFind(ids)\n    if variant in {\"merge_ultra\"}:\n        # Preserve current clusters, then add only extremely high-confidence\n        # cross-cluster dot-stack merges.\n        for members in by_cluster.values():\n            anchor = members[0]\n            for other in members[1:]:\n                uf.union(anchor, other)\n    else:\n        # Split current clusters by verified intra-cluster dot support.\n        for cluster, members in by_cluster.items():\n            if len(members) <= 1:\n                continue\n            g = pair_scores[\n                pair_scores[\"image_id_a\"].isin(members)\n                & pair_scores[\"image_id_b\"].isin(members)\n                & pair_scores[\"same_current_cluster\"].astype(bool)\n            ]\n            for row in g[g[\"score\"].astype(float) >= keep_thr].itertuples(index=False):\n                uf.union(int(row.image_id_a), int(row.image_id_b))\n\n    if merge_rank > 0:\n        cross = pair_scores[~pair_scores[\"same_current_cluster\"].astype(bool)].copy()\n        if not cross.empty:\n            cross = cross[\n                (cross[\"score\"].astype(float) >= merge_thr)\n                & (cross[\"overlap\"].astype(float) >= 0.13)\n                & (cross[\"point_score\"].astype(float) >= 0.36)\n                & (cross[\"stack_gain\"].astype(float) >= 0.43)\n            ].copy()\n            if not cross.empty:\n                neighbors: dict[int, list[tuple[int, float]]] = {i: [] for i in ids}\n                for row in cross.itertuples(index=False):\n                    a = int(row.image_id_a)\n                    b = int(row.image_id_b)\n                    s = float(row.score)\n                    neighbors[a].append((b, s))\n                    neighbors[b].append((a, s))\n                ranks: dict[tuple[int, int], int] = {}\n                for node, vals in neighbors.items():\n                    vals.sort(key=lambda x: -x[1])\n                    for rank, (other, _) in enumerate(vals, start=1):\n                        ranks[(node, other)] = rank\n                for row in cross.sort_values(\"score\", ascending=False).itertuples(index=False):\n                    a = int(row.image_id_a)\n                    b = int(row.image_id_b)\n                    if current_by_id[a] == current_by_id[b]:\n                        continue\n                    if ranks.get((a, b), 999) <= merge_rank and ranks.get((b, a), 999) <= merge_rank:\n                        uf.union(a, b)\n    components: dict[int, list[int]] = {}\n    for image_id in ids:\n        components.setdefault(uf.find(image_id), []).append(image_id)\n    current_sets = {cluster: set(members) for cluster, members in by_cluster.items()}\n    comp_order: dict[int, int] = {}\n    labels: dict[int, str] = {}\n    for root, members in sorted(components.items(), key=lambda kv: min(kv[1])):\n        member_set = set(members)\n        member_current = {current_by_id[i] for i in members}\n        if len(member_current) == 1:\n            current_cluster = next(iter(member_current))\n            if member_set == current_sets.get(current_cluster, set()):\n                for image_id in members:\n                    labels[image_id] = current_cluster\n                continue\n        comp_order[root] = len(comp_order)\n        new_label = f\"cluster_TexasHornedLizards_astrodot_{variant}_{comp_order[root]}\"\n        for image_id in members:\n            labels[image_id] = new_label\n    return labels\n\n\ndef summarize_texas_variant(labels: dict[int, str], pair_scores: pd.DataFrame, variant: str, current_labels: dict[int, str]) -> dict:\n    counts = pd.Series(list(labels.values())).value_counts()\n    changed = int(sum(1 for i, lab in labels.items() if lab != current_labels.get(i)))\n    return {\n        \"variant\": variant,\n        \"species\": TEXAS,\n        \"n_images\": int(len(labels)),\n        \"n_clusters\": int(counts.shape[0]),\n        \"singletons\": int((counts == 1).sum()),\n        \"max_cluster_size\": int(counts.max()) if not counts.empty else 0,\n        \"rows_changed_vs_current\": changed,\n        \"pair_score_mean\": float(pair_scores[\"score\"].mean()) if not pair_scores.empty else 0.0,\n        \"pair_score_p95\": float(pair_scores[\"score\"].quantile(0.95)) if not pair_scores.empty else 0.0,\n        \"same_current_mean\": float(pair_scores.loc[pair_scores[\"same_current_cluster\"].astype(bool), \"score\"].mean())\n        if not pair_scores.empty and pair_scores[\"same_current_cluster\"].astype(bool).any()\n        else 0.0,\n        \"cross_current_p99\": float(pair_scores.loc[~pair_scores[\"same_current_cluster\"].astype(bool), \"score\"].quantile(0.99))\n        if not pair_scores.empty and (~pair_scores[\"same_current_cluster\"].astype(bool)).any()\n        else 0.0,\n    }\n\n\ndef root_sift(desc: np.ndarray | None) -> np.ndarray | None:\n    if desc is None or len(desc) == 0:\n        return desc\n    desc = desc.astype(np.float32)\n    desc /= np.maximum(1e-7, desc.sum(axis=1, keepdims=True))\n    return np.sqrt(desc).astype(np.float32)\n\n\ndef extract_local_features(row: dict, species: str, max_side: int, detector) -> tuple[list, np.ndarray | None, np.ndarray]:\n    cfg = core.SPECIES_CONFIGS[species]\n    rgb, mask = core.read_rgb_with_optional_mask(row.get(\"view_path\", row.get(\"source_path\")), row.get(\"mask_path\"), max_side)\n    if mask is None:\n        mask = core.estimate_foreground_mask(rgb)\n    roi_rgb, roi_mask, _ = core.species_roi(\n        rgb,\n        species,\n        str(row.get(\"orientation\", \"unknown\")).lower(),\n        cfg,\n        mask_override=mask,\n    )\n    gray = core.pattern_gray(roi_rgb, species)\n    kps, desc = detector.detectAndCompute(gray, roi_mask)\n    if kps is None:\n        kps = []\n    desc = root_sift(desc)\n    return kps, desc, roi_mask\n\n\ndef local_pair_score(a_feat, b_feat, species: str) -> dict:\n    kps_a, desc_a, mask_a = a_feat\n    kps_b, desc_b, mask_b = b_feat\n    if desc_a is None or desc_b is None or len(desc_a) < 4 or len(desc_b) < 4:\n        return {\"score\": 0.0, \"inliers\": 0, \"good_matches\": 0, \"inlier_ratio\": 0.0}\n    matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)\n    try:\n        knn = matcher.knnMatch(desc_a, desc_b, k=2)\n    except Exception:\n        return {\"score\": 0.0, \"inliers\": 0, \"good_matches\": 0, \"inlier_ratio\": 0.0}\n    ratio = 0.80 if species == \"SalamanderID2025\" else 0.76\n    good = []\n    for pair in knn:\n        if len(pair) < 2:\n            continue\n        m, n = pair\n        if m.distance < ratio * n.distance:\n            good.append(m)\n    if len(good) < 4:\n        return {\"score\": 0.0, \"inliers\": 0, \"good_matches\": len(good), \"inlier_ratio\": 0.0}\n    pts_a = np.float32([kps_a[m.queryIdx].pt for m in good]).reshape(-1, 2)\n    pts_b = np.float32([kps_b[m.trainIdx].pt for m in good]).reshape(-1, 2)\n    _, inlier_mask = cv2.estimateAffinePartial2D(\n        pts_a,\n        pts_b,\n        method=cv2.RANSAC,\n        ransacReprojThreshold=5.0,\n        maxIters=1600,\n        confidence=0.995,\n    )\n    inliers = int(inlier_mask.reshape(-1).sum()) if inlier_mask is not None else 0\n    denom = max(1, min(len(kps_a), len(kps_b), len(good)))\n    inlier_ratio = float(inliers / denom)\n    inlier_term = min(1.0, inliers / (28.0 if species != \"SalamanderID2025\" else 18.0))\n    ratio_term = min(1.0, inlier_ratio / 0.25)\n    match_term = min(1.0, len(good) / 60.0)\n    score = 0.55 * inlier_term + 0.30 * ratio_term + 0.15 * match_term\n    return {\"score\": float(score), \"inliers\": inliers, \"good_matches\": int(len(good)), \"inlier_ratio\": inlier_ratio}\n\n\ndef sample_train_rows(rows: pd.DataFrame, max_images: int, max_per_identity: int) -> pd.DataFrame:\n    labeled = rows[rows[\"identity\"].fillna(\"\").astype(str).str.len().gt(0)].copy()\n    if labeled.empty:\n        return labeled\n    rng = random.Random(SEED)\n    chosen = []\n    for _, group in labeled.groupby(\"identity\"):\n        recs = group.sort_values(\"image_id\").to_dict(\"records\")\n        if len(recs) > max_per_identity:\n            recs = rng.sample(recs, max_per_identity)\n        chosen.extend(recs)\n    if max_images and len(chosen) > max_images:\n        chosen = rng.sample(chosen, max_images)\n    chosen.sort(key=lambda r: int(r[\"image_id\"]))\n    return pd.DataFrame(chosen)\n\n\ndef sample_validation_pairs(items: list[dict], budget: int) -> list[tuple[int, int]]:\n    rng = random.Random(SEED)\n    ids = [int(r[\"image_id\"]) for r in items]\n    identities = {int(r[\"image_id\"]): str(r.get(\"identity\", \"\")) for r in items}\n    by_identity: dict[str, list[int]] = {}\n    for i in ids:\n        by_identity.setdefault(identities[i], []).append(i)\n    positives = []\n    for members in by_identity.values():\n        if len(members) < 2:\n            continue\n        combos = list(itertools.combinations(sorted(members), 2))\n        if len(combos) > 10:\n            combos = rng.sample(combos, 10)\n        positives.extend(combos)\n    if len(positives) > budget // 2:\n        positives = rng.sample(positives, max(1, budget // 2))\n    negatives = set()\n    target_neg = min(budget - len(positives), max(600 if positives else budget, len(positives) * 2))\n    attempts = 0\n    while len(negatives) < target_neg and attempts < max(100, target_neg * 35) and len(ids) >= 2:\n        a, b = rng.sample(ids, 2)\n        attempts += 1\n        if identities[a] == identities[b]:\n            continue\n        negatives.add((a, b) if a < b else (b, a))\n    pairs = positives + sorted(negatives)\n    if len(pairs) > budget:\n        pairs = rng.sample(pairs, budget)\n    return [(int(a), int(b)) for a, b in pairs]\n\n\ndef auc_rank(y_true: np.ndarray, scores: np.ndarray) -> float:\n    y = y_true.astype(bool)\n    n_pos = int(y.sum())\n    n_neg = int((~y).sum())\n    if n_pos == 0 or n_neg == 0:\n        return float(\"nan\")\n    order = np.argsort(scores)\n    ranks = np.empty_like(order, dtype=np.float64)\n    ranks[order] = np.arange(1, len(scores) + 1, dtype=np.float64)\n    pos_rank_sum = float(ranks[y].sum())\n    return float((pos_rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))\n\n\ndef validate_reused_species(metadata: pd.DataFrame, args: argparse.Namespace, reports_dir: Path) -> pd.DataFrame:\n    rows_out = []\n    try:\n        detector = cv2.SIFT_create(nfeatures=1100, contrastThreshold=0.012, edgeThreshold=12)\n    except Exception:\n        print(\"[warn] SIFT unavailable; reused-species validation skipped.\")\n        return pd.DataFrame()\n    train_all = metadata[metadata[\"split\"].eq(\"train\")].copy()\n    for species in REUSED_SPECIES:\n        print(f\"[{species}] 2025-style local validation\")\n        train_rows = train_all[train_all[\"species_id\"].eq(species)].copy()\n        train_rows = sample_train_rows(train_rows, args.reused_max_train_images, args.reused_max_per_identity)\n        if args.smoke:\n            train_rows = train_rows.head(44)\n        records = train_rows.to_dict(\"records\")\n        if not records:\n            continue\n        feats = {}\n        failures = 0\n        for idx, row in enumerate(records, start=1):\n            try:\n                feats[int(row[\"image_id\"])] = extract_local_features(row, species, args.max_side, detector)\n            except Exception as exc:\n                failures += 1\n                feats[int(row[\"image_id\"])] = ([], None, np.zeros((8, 8), dtype=np.uint8))\n                print(f\"[warn] {species} feature fail image_id={row.get('image_id')}: {exc}\")\n            if idx % 100 == 0:\n                print(f\"[{species}] local features {idx}/{len(records)}\")\n        pairs = sample_validation_pairs(records, min(args.reused_train_pair_budget, 350 if args.smoke else args.reused_train_pair_budget))\n        score_rows = []\n        identity_by_id = {int(r[\"image_id\"]): str(r.get(\"identity\", \"\")) for r in records}\n        for a, b in pairs:\n            s = local_pair_score(feats[a], feats[b], species)\n            same = bool(identity_by_id[a] and identity_by_id[a] == identity_by_id[b])\n            score_rows.append({\"species\": species, \"image_id_a\": a, \"image_id_b\": b, \"same_identity\": same, **s})\n        pair_df = pd.DataFrame(score_rows)\n        pair_df.to_csv(reports_dir / f\"{VERSION}_{species}_train_local_pair_scores.csv\", index=False)\n        if pair_df.empty:\n            continue\n        y = pair_df[\"same_identity\"].astype(bool).to_numpy()\n        scores = pair_df[\"score\"].astype(float).to_numpy()\n        row = {\n            \"species\": species,\n            \"n_pairs\": int(len(pair_df)),\n            \"n_positive\": int(y.sum()),\n            \"n_negative\": int((~y).sum()),\n            \"auc\": auc_rank(y, scores),\n            \"feature_failures\": int(failures),\n            \"median_inliers_positive\": float(pair_df.loc[pair_df[\"same_identity\"], \"inliers\"].median()) if y.any() else 0.0,\n            \"median_inliers_negative\": float(pair_df.loc[~pair_df[\"same_identity\"], \"inliers\"].median()) if (~y).any() else 0.0,\n        }\n        for thr in [0.35, 0.45, 0.55, 0.65]:\n            pred = scores >= thr\n            tp = int((pred & y).sum())\n            fp = int((pred & (~y)).sum())\n            fn = int(((~pred) & y).sum())\n            row[f\"precision_at_{thr:.2f}\"] = float(tp / max(1, tp + fp))\n            row[f\"recall_at_{thr:.2f}\"] = float(tp / max(1, tp + fn))\n            row[f\"accepted_at_{thr:.2f}\"] = int(pred.sum())\n        rows_out.append(row)\n    return pd.DataFrame(rows_out)\n\n\ndef build_submission(current: pd.DataFrame, test_rows: pd.DataFrame, texas_labels: dict[int, str], out_path: Path) -> pd.DataFrame:\n    sub = current.copy()\n    texas_ids = set(test_rows.loc[test_rows[\"species_id\"].eq(TEXAS), \"image_id\"].astype(int).tolist())\n    current_map = dict(zip(sub[\"image_id\"].astype(int), sub[\"cluster\"].astype(str)))\n    sub[\"cluster\"] = sub[\"image_id\"].astype(int).map(\n        lambda i: texas_labels.get(i, current_map[i]) if i in texas_ids else current_map[i]\n    )\n    sub.to_csv(out_path, index=False)\n    return sub\n\n\ndef summarize_submission(sub: pd.DataFrame, current: pd.DataFrame, test_rows: pd.DataFrame, variant: str) -> list[dict]:\n    cur_map = dict(zip(current[\"image_id\"].astype(int), current[\"cluster\"].astype(str)))\n    sub_map = dict(zip(sub[\"image_id\"].astype(int), sub[\"cluster\"].astype(str)))\n    rows = []\n    for species in core.SPECIES_ORDER:\n        ids = test_rows.loc[test_rows[\"species_id\"].eq(species), \"image_id\"].astype(int).tolist()\n        labels = [sub_map[i] for i in ids]\n        counts = pd.Series(labels).value_counts()\n        rows.append(\n            {\n                \"variant\": variant,\n                \"species\": species,\n                \"n_images\": int(len(ids)),\n                \"n_clusters\": int(counts.shape[0]),\n                \"singletons\": int((counts == 1).sum()),\n                \"max_cluster_size\": int(counts.max()) if not counts.empty else 0,\n                \"rows_changed_vs_current\": int(sum(1 for i in ids if sub_map[i] != cur_map[i])),\n            }\n        )\n    return rows\n\n\ndef heat_to_image(heat: np.ndarray, mask: np.ndarray) -> Image.Image:\n    h = np.clip(heat, 0, 1)\n    arr = np.zeros((h.shape[0], h.shape[1], 3), dtype=np.uint8)\n    arr[:, :, 0] = np.clip(h * 255, 0, 255).astype(np.uint8)\n    arr[:, :, 1] = np.clip((1.0 - h) * 120, 0, 255).astype(np.uint8)\n    arr[:, :, 2] = np.clip((1.0 - h) * 90, 0, 255).astype(np.uint8)\n    arr[mask == 0] = BACKGROUND\n    return Image.fromarray(arr)\n\n\ndef draw_texas_item(item: TexasDotItem, size: tuple[int, int]) -> Image.Image:\n    rgb = item.belly_rgb.copy()\n    mask = item.belly_mask\n    dim = (rgb.astype(np.float32) * 0.40 + BACKGROUND.astype(np.float32) * 0.60).astype(np.uint8)\n    rgb[mask == 0] = dim[mask == 0]\n    contours, _ = cv2.findContours(np.where(mask > 0, 255, 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    cv2.drawContours(rgb, contours, -1, (30, 255, 70), 2)\n    h, w = mask.shape[:2]\n    for p in item.dot_points[:160]:\n        x = int(round(float(p[0]) * max(1, w - 1)))\n        y = int(round(float(p[1]) * max(1, h - 1)))\n        cv2.circle(rgb, (x, y), 2, (255, 30, 30), 1, cv2.LINE_AA)\n    img = Image.fromarray(rgb)\n    img.thumbnail(size, Image.Resampling.BILINEAR)\n    canvas = Image.new(\"RGB\", size, (18, 18, 18))\n    canvas.paste(img, ((size[0] - img.width) // 2, (size[1] - img.height) // 2))\n    return canvas\n\n\ndef thumb(path: str, size: tuple[int, int]) -> Image.Image:\n    try:\n        img = Image.open(path).convert(\"RGB\")\n    except Exception:\n        img = Image.new(\"RGB\", size, (25, 25, 25))\n    img.thumbnail(size, Image.Resampling.BILINEAR)\n    canvas = Image.new(\"RGB\", size, (18, 18, 18))\n    canvas.paste(img, ((size[0] - img.width) // 2, (size[1] - img.height) // 2))\n    return canvas\n\n\ndef save_texas_preview(items: list[TexasDotItem], out_path: Path, limit: int) -> None:\n    chosen = items[:limit]\n    if not chosen:\n        return\n    tile_w, tile_h = 190, 190\n    label_h = 22\n    cols = 4\n    canvas = Image.new(\"RGB\", (cols * tile_w, len(chosen) * (tile_h + label_h)), (18, 18, 18))\n    draw = ImageDraw.Draw(canvas)\n    for r, item in enumerate(chosen):\n        y = r * (tile_h + label_h)\n        panels = [\n            thumb(item.source_path, (tile_w, tile_h)),\n            thumb(item.view_path, (tile_w, tile_h)),\n            draw_texas_item(item, (tile_w, tile_h)),\n            heat_to_image(item.dot_heat, item.belly_mask),\n        ]\n        labels = [f\"orig {item.image_id}\", item.view_source[:20], \"belly dots\", \"dot heat\"]\n        for c, panel in enumerate(panels):\n            panel = panel.copy()\n            panel.thumbnail((tile_w, tile_h), Image.Resampling.BILINEAR)\n            x = c * tile_w\n            draw.text((x + 5, y + 4), labels[c], fill=(245, 240, 145))\n            canvas.paste(panel, (x + (tile_w - panel.width) // 2, y + label_h + (tile_h - panel.height) // 2))\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    canvas.save(out_path, quality=92)\n\n\ndef save_pair_preview(items: list[TexasDotItem], pair_scores: pd.DataFrame, out_path: Path, limit: int, cross_only: bool = False) -> None:\n    if pair_scores.empty:\n        return\n    g = pair_scores.copy()\n    if cross_only:\n        g = g[~g[\"same_current_cluster\"].astype(bool)]\n    g = g.sort_values(\"score\", ascending=False).head(limit)\n    if g.empty:\n        return\n    by_id = {it.image_id: it for it in items}\n    panel_w, panel_h = 840, 260\n    rows = []\n    for edge in g.to_dict(\"records\"):\n        a = by_id.get(int(edge[\"image_id_a\"]))\n        b = by_id.get(int(edge[\"image_id_b\"]))\n        if a is None or b is None:\n            continue\n        pa = heat_to_image(a.dot_heat, a.belly_mask)\n        pb = heat_to_image(b.dot_heat, b.belly_mask)\n        pa.thumbnail((300, 220), Image.Resampling.BILINEAR)\n        pb.thumbnail((300, 220), Image.Resampling.BILINEAR)\n        row_img = Image.new(\"RGB\", (panel_w, panel_h), (18, 18, 18))\n        draw = ImageDraw.Draw(row_img)\n        text = (\n            f\"{a.image_id} vs {b.image_id} score={float(edge['score']):.3f} \"\n            f\"corr={float(edge['corr']):.3f} pts={float(edge['point_score']):.3f} \"\n            f\"stack={float(edge['stack_gain']):.3f} same_current={edge['same_current_cluster']}\"\n        )\n        draw.text((6, 5), text, fill=(255, 240, 120))\n        row_img.paste(pa, (12, 35))\n        row_img.paste(pb, (440, 35))\n        rows.append(row_img)\n    canvas = Image.new(\"RGB\", (panel_w, panel_h * len(rows)), (18, 18, 18))\n    for idx, row in enumerate(rows):\n        canvas.paste(row, (0, idx * panel_h))\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    canvas.save(out_path, quality=92)\n\n\ndef main() -> None:\n    random.seed(SEED)\n    np.random.seed(SEED)\n    args = parse_args()\n    if args.smoke:\n        args.texas_pair_budget = min(args.texas_pair_budget or 600, 600)\n        args.reused_train_pair_budget = min(args.reused_train_pair_budget, 260)\n        args.reused_max_train_images = min(args.reused_max_train_images, 48)\n        args.save_visualizations = True\n\n    data_root = find_data_root(args.data_root)\n    sam_manifest = find_sam_manifest(args.sam_manifest)\n    metadata, manifest_info = prepare_metadata(data_root, sam_manifest)\n    metadata = metadata[metadata[\"species_id\"].isin(core.SPECIES_ORDER)].copy()\n    test_rows = metadata[metadata[\"split\"].eq(\"test\")].copy()\n    output_root = Path(args.output_root) if args.output_root else Path.cwd() / f\"animalclef_{VERSION}\"\n    reports_dir = output_root / \"reports\"\n    sub_dir = output_root / \"submissions\"\n    viz_dir = output_root / \"visualizations\"\n    reports_dir.mkdir(parents=True, exist_ok=True)\n    sub_dir.mkdir(parents=True, exist_ok=True)\n    if args.save_visualizations:\n        viz_dir.mkdir(parents=True, exist_ok=True)\n\n    current_best_path = find_current_best(args.current_best_submission, data_root)\n    current = core.load_submission(current_best_path)\n    current_labels = dict(zip(current[\"image_id\"].astype(int), current[\"cluster\"].astype(str)))\n\n    print(f\"VERSION={VERSION}\")\n    print(f\"data_root={data_root}\")\n    print(f\"sam_manifest={sam_manifest}\")\n    print(f\"current_best={current_best_path}\")\n    print(f\"output_root={output_root}\")\n    print(json.dumps(manifest_info, indent=2))\n\n    texas_rows = test_rows[test_rows[\"species_id\"].eq(TEXAS)].sort_values(\"image_id\").copy()\n    if args.smoke:\n        texas_rows = texas_rows.head(36)\n    texas_items: list[TexasDotItem] = []\n    for idx, row in enumerate(texas_rows.to_dict(\"records\"), start=1):\n        image_id = int(row[\"image_id\"])\n        try:\n            texas_items.append(texas_belly_template(row, current_labels[image_id], args))\n        except Exception as exc:\n            print(f\"[warn] Texas template failed image_id={image_id}: {exc}\")\n        if idx % 75 == 0:\n            print(f\"[Texas astro-dot] templates {idx}/{len(texas_rows)}\")\n\n    if args.save_visualizations:\n        save_texas_preview(texas_items, viz_dir / f\"{VERSION}_Texas_template_preview.jpg\", args.visual_limit)\n\n    texas_pair_scores = score_all_texas_pairs(texas_items, args.texas_pair_budget)\n    texas_pair_scores.to_csv(reports_dir / f\"{VERSION}_Texas_pair_scores.csv\", index=False)\n    if args.save_visualizations:\n        save_pair_preview(texas_items, texas_pair_scores, viz_dir / f\"{VERSION}_Texas_top_pairs_all.jpg\", max(5, args.visual_limit // 2), False)\n        save_pair_preview(texas_items, texas_pair_scores, viz_dir / f\"{VERSION}_Texas_top_pairs_cross_cluster.jpg\", max(5, args.visual_limit // 2), True)\n\n    reused_report = pd.DataFrame()\n    if not args.skip_reused_validation:\n        reused_report = validate_reused_species(metadata, args, reports_dir)\n        reused_report.to_csv(reports_dir / f\"{VERSION}_reused_species_2025style_validation.csv\", index=False)\n\n    variants = [\"split_strict\", \"merge_ultra\", \"splitmerge_guarded\", \"splitmerge_swing\"]\n    candidate_rows: list[dict] = []\n    texas_variant_rows: list[dict] = []\n    texas_current = {it.image_id: it.current_cluster for it in texas_items}\n    for variant in variants:\n        texas_labels = texas_variant_labels(texas_items, texas_pair_scores, variant)\n        texas_variant_rows.append(summarize_texas_variant(texas_labels, texas_pair_scores, variant, texas_current))\n        out_path = sub_dir / f\"submission_{VERSION}_{variant}.csv\"\n        sub = build_submission(current, test_rows, texas_labels, out_path)\n        candidate_rows.extend(summarize_submission(sub, current, test_rows, variant))\n        print(f\"wrote {out_path}\")\n\n    texas_variant_report = pd.DataFrame(texas_variant_rows)\n    candidate_report = pd.DataFrame(candidate_rows)\n    texas_variant_report.to_csv(reports_dir / f\"{VERSION}_texas_variant_report.csv\", index=False)\n    candidate_report.to_csv(reports_dir / f\"{VERSION}_candidate_report.csv\", index=False)\n\n    summary = {\n        \"version\": VERSION,\n        \"data_root\": str(data_root),\n        \"sam_manifest\": str(sam_manifest) if sam_manifest else None,\n        \"current_best\": str(current_best_path),\n        \"manifest_info\": manifest_info,\n        \"outputs\": {\n            \"reports_dir\": str(reports_dir),\n            \"submissions_dir\": str(sub_dir),\n            \"visualizations_dir\": str(viz_dir),\n        },\n        \"texas_items\": int(len(texas_items)),\n        \"texas_pairs\": int(len(texas_pair_scores)),\n        \"texas_variant_report\": texas_variant_report.to_dict(\"records\"),\n        \"candidate_report\": candidate_report.to_dict(\"records\"),\n        \"reused_validation\": reused_report.to_dict(\"records\") if not reused_report.empty else [],\n    }\n    (reports_dir / f\"{VERSION}_summary.json\").write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n    print(\"\\nTexas variant report:\")\n    print(texas_variant_report.to_string(index=False))\n    print(\"\\nCandidate report:\")\n    print(candidate_report.to_string(index=False))\n    if not reused_report.empty:\n        print(\"\\nReused-species 2025-style validation:\")\n        print(reused_report.to_string(index=False))\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
Path("texas_astrodot_boundary_suppressed_v20260429.py").write_text("from __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nimport texas_astrodot_2025reuse_v20260426 as base\n\n\nVERSION = \"texas_astrodot_boundary_suppressed_v20260429\"\nTEXAS = base.TEXAS\n\nCURRENT_BEST_FILENAMES = [\n    \"submission_032583_salamander_p80_cluster_cap_v20260429.csv\",\n    \"submission_032368_gamble_salamander_p80_cap_v20260429.csv\",\n    \"submission.csv\",\n]\n\n\ndef find_current_best_boundary(user_value: str | None, data_root: Path) -> Path:\n    if user_value:\n        p = Path(user_value)\n        if p.exists():\n            return p.resolve()\n    roots = [\n        Path(\"/kaggle/input\"),\n        Path(\"/kaggle/working\"),\n        Path.cwd(),\n        Path.cwd().parent,\n        data_root.parent,\n        Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\current_wildfusion_graph_v20260423\\submissions\"),\n        Path(r\"C:\\Users\\Hanif\\Documents\\kaggle\\AnimalCLEF2026\\current_wildfusion_graph_v20260423\"),\n    ]\n    matches: list[Path] = []\n    for root in roots:\n        if not root.exists():\n            continue\n        for filename in CURRENT_BEST_FILENAMES:\n            direct = root / filename\n            if direct.exists():\n                matches.append(direct)\n            try:\n                matches.extend(root.rglob(filename))\n            except Exception:\n                continue\n\n    def preference(path: Path) -> tuple[int, int, str]:\n        text = str(path).replace(\"\\\\\", \"/\").lower()\n        penalty = 0\n        if \"032583\" in text or \"032368_gamble\" in text:\n            penalty -= 40\n        if \"lb032583\" in text or \"salamander_p80\" in text:\n            penalty -= 20\n        if text.endswith(\"/submission.csv\"):\n            penalty += 5\n        if \"local_smoke\" in text:\n            penalty += 50\n        return penalty, -int(path.stat().st_size), text\n\n    matches = sorted({p.resolve() for p in matches}, key=preference)\n    if matches:\n        return matches[0]\n    return base.find_current_best(user_value, data_root)\n\n\ndef align_vertical_no_flip(rgb: np.ndarray, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray, dict]:\n    \"\"\"Align the ventral lizard body but do not flip: field heads are already up.\"\"\"\n    crop_rgb, crop_mask = base.crop_to_mask(rgb, mask, 0.08)\n    angle = base.core.pca_angle_degrees(crop_mask)\n    rotate_angle = 90.0 - angle\n    if abs(rotate_angle) > 1.5:\n        crop_rgb, crop_mask = base.core.rotate_bound(crop_rgb, crop_mask, rotate_angle)\n        crop_rgb, crop_mask = base.crop_to_mask(crop_rgb, crop_mask, 0.04)\n    h, _ = crop_mask.shape[:2]\n    widths = []\n    for yf in [0.18, 0.30, 0.70, 0.84]:\n        y = int(np.clip(round(h * yf), 0, h - 1))\n        xs = np.where(crop_mask[y] > 0)[0]\n        widths.append(float(xs.max() - xs.min() + 1) if len(xs) else 0.0)\n    return crop_rgb, crop_mask, {\n        \"pca_angle\": float(angle),\n        \"rotate_angle\": float(rotate_angle),\n        \"top_width\": float(max(widths[:2])),\n        \"bottom_width\": float(max(widths[2:])),\n        \"flipped_vertical\": False,\n        \"orientation_rule\": \"head_already_top_no_flip\",\n    }\n\n\ndef interior_weight_from_mask(mask: np.ndarray) -> np.ndarray:\n    \"\"\"Softly suppress the body outline while preserving full body geometry.\n\n    The old oval mask was too rigid; the no-oval mask kept useful peripheral\n    dots but also allowed the animal contour to dominate. This distance-weighted\n    formula keeps the real mask shape and scale but makes the contour band weak:\n\n        W = 0.08 + 0.92 * clip((D - r0) / (r1 - r0), 0, 1) ** gamma\n\n    where D is distance to the foreground boundary. Dots near the center keep\n    full strength; the hard silhouette edge is downweighted, not cropped away.\n    \"\"\"\n    m = np.where(mask > 0, 255, 0).astype(np.uint8)\n    h, w = m.shape[:2]\n    if int((m > 0).sum()) < 40:\n        return np.ones((h, w), dtype=np.float32)\n    dist = cv2.distanceTransform(m, cv2.DIST_L2, 5).astype(np.float32)\n    short = float(max(1, min(h, w)))\n    r0 = max(2.0, short * 0.020)\n    r1 = max(r0 + 2.0, short * 0.115)\n    core = np.clip((dist - r0) / max(1e-6, r1 - r0), 0.0, 1.0)\n    weight = 0.08 + 0.92 * np.power(core, 0.72)\n    weight[m == 0] = 0.0\n    return weight.astype(np.float32)\n\n\ndef boundary_suppressed_dot_heat(\n    belly_rgb: np.ndarray,\n    belly_mask: np.ndarray,\n    interior_weight: np.ndarray,\n) -> np.ndarray:\n    lab = cv2.cvtColor(belly_rgb, cv2.COLOR_RGB2LAB)\n    l_eq = base.clahe_u8(lab[:, :, 0])\n    dark = (255.0 - l_eq.astype(np.float32)) / 255.0\n    blackhat = cv2.morphologyEx(l_eq, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13)))\n    raw = 0.58 * (blackhat.astype(np.float32) / 255.0) + 0.42 * dark\n    valid = belly_mask > 0\n    if int(valid.sum()) > 20:\n        vals = raw[valid]\n        lo = float(np.percentile(vals, 12))\n        hi = float(np.percentile(vals, 98))\n        raw = (raw - lo) / max(1e-6, hi - lo)\n    raw = np.clip(raw, 0.0, 1.0)\n\n    edge = cv2.morphologyEx(\n        np.where(valid, 255, 0).astype(np.uint8),\n        cv2.MORPH_GRADIENT,\n        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)),\n    ).astype(np.float32) / 255.0\n    edge = cv2.GaussianBlur(edge, (9, 9), 0)\n\n    heat = raw * interior_weight\n    heat = np.clip(heat - 0.20 * edge, 0.0, 1.0)\n    heat[~valid] = 0.0\n    heat = cv2.GaussianBlur(heat, (3, 3), 0)\n    return heat.astype(np.float32)\n\n\ndef weighted_dot_points_from_heat(heat: np.ndarray, mask: np.ndarray, interior_weight: np.ndarray) -> np.ndarray:\n    valid = (mask > 0) & (interior_weight >= 0.16)\n    if int(valid.sum()) < 40:\n        valid = mask > 0\n    vals = heat[valid]\n    if len(vals) < 40:\n        return np.zeros((0, 4), dtype=np.float32)\n    thr = max(float(np.percentile(vals, 84)), float(vals.mean() + 0.55 * vals.std()))\n    binary = np.zeros(heat.shape, dtype=np.uint8)\n    binary[(heat >= thr) & valid] = 255\n    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))\n    n, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, 8)\n    h, w = heat.shape[:2]\n    total = float(h * w)\n    pts: list[list[float]] = []\n    min_area = max(2.0, total * 0.00004)\n    max_area = total * 0.010\n    for idx in range(1, n):\n        area = float(stats[idx, cv2.CC_STAT_AREA])\n        if area < min_area or area > max_area:\n            continue\n        bw = max(1.0, float(stats[idx, cv2.CC_STAT_WIDTH]))\n        bh = max(1.0, float(stats[idx, cv2.CC_STAT_HEIGHT]))\n        if max(bw / bh, bh / bw) > 4.8:\n            continue\n        cx, cy = centroids[idx]\n        comp = labels == idx\n        strength = float((heat[comp] * np.maximum(0.2, interior_weight[comp])).mean())\n        pts.append([float(cx) / max(1, w - 1), float(cy) / max(1, h - 1), area / total, strength])\n    if not pts:\n        return np.zeros((0, 4), dtype=np.float32)\n    pts_arr = np.asarray(pts, dtype=np.float32)\n    order = np.argsort(-pts_arr[:, 3])\n    return pts_arr[order[:260]]\n\n\ndef texas_belly_template_boundary(row: dict, current_cluster: str, args: argparse.Namespace) -> base.TexasDotItem:\n    \"\"\"Full aligned crop, no oval, but contour-suppressed belly-dot signal.\"\"\"\n    rgb, mask, quality = base.read_rgb_mask(row, args.max_side)\n    aligned_rgb, aligned_mask, debug = align_vertical_no_flip(rgb, mask)\n    w = int(args.texas_canvas_w)\n    h = int(args.texas_canvas_h)\n    belly_rgb = cv2.resize(aligned_rgb, (w, h), interpolation=cv2.INTER_AREA)\n    belly_mask_full = cv2.resize(aligned_mask, (w, h), interpolation=cv2.INTER_NEAREST)\n    belly_mask_full = base.clean_mask(belly_mask_full)\n\n    coverage = float(belly_mask_full.mean() / 255.0)\n    if coverage < 0.035:\n        belly_mask_full = np.full((h, w), 255, dtype=np.uint8)\n        quality *= 0.68\n        coverage = 1.0\n    elif coverage > 0.92:\n        quality *= 0.94\n\n    interior_weight = interior_weight_from_mask(belly_mask_full)\n    effective_mask = np.where((belly_mask_full > 0) & (interior_weight >= 0.12), 255, 0).astype(np.uint8)\n    if float(effective_mask.mean() / 255.0) < 0.025:\n        effective_mask = belly_mask_full.copy()\n        interior_weight = np.where(effective_mask > 0, 1.0, 0.0).astype(np.float32)\n        quality *= 0.74\n\n    heat = boundary_suppressed_dot_heat(belly_rgb, belly_mask_full, interior_weight)\n    points = weighted_dot_points_from_heat(heat, effective_mask, interior_weight)\n    small = cv2.resize(heat, (32, 48), interpolation=cv2.INTER_AREA).reshape(-1)\n    weight_small = cv2.resize(interior_weight, (32, 48), interpolation=cv2.INTER_AREA).reshape(-1)\n    mask_small = cv2.resize((effective_mask > 0).astype(np.float32), (32, 48), interpolation=cv2.INTER_AREA).reshape(-1)\n    vector = base.normalize(np.concatenate([small * weight_small, mask_small * 0.10, weight_small * 0.08]).astype(np.float32))\n    quality *= min(1.0, 0.70 + 0.30 * min(1.0, len(points) / 42.0))\n\n    debug = dict(debug)\n    debug[\"mask_mode\"] = \"full_aligned_crop_boundary_suppressed\"\n    debug[\"full_mask_coverage\"] = coverage\n    debug[\"effective_mask_coverage\"] = float(effective_mask.mean() / 255.0)\n    debug[\"mean_interior_weight\"] = float(interior_weight[belly_mask_full > 0].mean()) if (belly_mask_full > 0).any() else 0.0\n\n    return base.TexasDotItem(\n        image_id=int(row[\"image_id\"]),\n        current_cluster=str(current_cluster),\n        source_path=str(row.get(\"source_path\", \"\")),\n        view_path=str(row.get(\"view_path\", row.get(\"source_path\", \"\"))),\n        view_source=str(row.get(\"view_source\", \"\")),\n        belly_rgb=belly_rgb,\n        belly_mask=effective_mask,\n        dot_heat=heat,\n        dot_points=points,\n        vector=vector,\n        quality=float(quality),\n        debug=debug,\n    )\n\n\ndef texas_pair_score_boundary(a: base.TexasDotItem, b: base.TexasDotItem) -> dict:\n    \"\"\"Dot-centered score: point alignment and stack sharpness matter more than contour correlation.\"\"\"\n    desc_cos = float(np.dot(a.vector, b.vector))\n    h, w = a.dot_heat.shape[:2]\n    best = {\n        \"score\": 0.0,\n        \"corr\": 0.0,\n        \"overlap\": 0.0,\n        \"stack_gain\": 0.0,\n        \"point_score\": 0.0,\n        \"descriptor_cosine\": desc_cos,\n        \"dx\": 0,\n        \"dy\": 0,\n        \"transform\": \"identity\",\n    }\n    common_mask = np.where(a.belly_mask > 0, 1.0, 0.0).astype(np.float32)\n    base_dx, base_dy = base.phase_shift(a.dot_heat, b.dot_heat, common_mask)\n    candidates = {(0, 0), (base_dx, base_dy)}\n    for dx0, dy0 in [(base_dx, base_dy), (0, 0)]:\n        for ddx in (-8, 0, 8):\n            for ddy in (-8, 0, 8):\n                candidates.add((int(np.clip(dx0 + ddx, -24, 24)), int(np.clip(dy0 + ddy, -24, 24))))\n    for dx, dy in candidates:\n        corr, overlap, stack_gain = base.masked_corr_and_stack(a.dot_heat, a.belly_mask, b.dot_heat, b.belly_mask, dx, dy)\n        if overlap < 0.040:\n            continue\n        point_score = base.chamfer_dot_score(a.dot_points, b.dot_points, dx / max(1, w - 1), dy / max(1, h - 1))\n        fused = 0.25 * corr + 0.42 * point_score + 0.25 * stack_gain + 0.08 * max(0.0, desc_cos)\n        if min(len(a.dot_points), len(b.dot_points)) < 12:\n            fused *= 0.82\n        fused *= min(a.quality, b.quality)\n        if fused > best[\"score\"]:\n            best = {\n                \"score\": float(fused),\n                \"corr\": float(corr),\n                \"overlap\": float(overlap),\n                \"stack_gain\": float(stack_gain),\n                \"point_score\": float(point_score),\n                \"descriptor_cosine\": desc_cos,\n                \"dx\": int(dx),\n                \"dy\": int(dy),\n                \"transform\": \"identity\",\n            }\n    best[\"points_a\"] = int(len(a.dot_points))\n    best[\"points_b\"] = int(len(b.dot_points))\n    best[\"quality_a\"] = float(a.quality)\n    best[\"quality_b\"] = float(b.quality)\n    best[\"same_current_cluster\"] = bool(a.current_cluster == b.current_cluster)\n    return best\n\n\ndef texas_variant_labels_boundary(items: list[base.TexasDotItem], pair_scores: pd.DataFrame, variant: str) -> dict[int, str]:\n    ids = [it.image_id for it in items]\n    by_cluster: dict[str, list[int]] = {}\n    current_by_id = {it.image_id: it.current_cluster for it in items}\n    for it in items:\n        by_cluster.setdefault(it.current_cluster, []).append(it.image_id)\n\n    if variant == \"split_strict\":\n        keep_thr, merge_thr, merge_rank = 0.330, 9.9, 0\n        max_cross_component = 999\n    elif variant == \"merge_ultra\":\n        keep_thr, merge_thr, merge_rank = 0.0, 0.575, 1\n        max_cross_component = 32\n    elif variant == \"splitmerge_guarded\":\n        keep_thr, merge_thr, merge_rank = 0.305, 0.555, 1\n        max_cross_component = 32\n    else:\n        keep_thr, merge_thr, merge_rank = 0.285, 0.540, 1\n        max_cross_component = 32\n\n    uf = base.UnionFind(ids)\n    if variant == \"merge_ultra\":\n        for members in by_cluster.values():\n            anchor = members[0]\n            for other in members[1:]:\n                uf.union(anchor, other)\n    else:\n        for members in by_cluster.values():\n            if len(members) <= 1:\n                continue\n            g = pair_scores[\n                pair_scores[\"image_id_a\"].isin(members)\n                & pair_scores[\"image_id_b\"].isin(members)\n                & pair_scores[\"same_current_cluster\"].astype(bool)\n            ]\n            for row in g[g[\"score\"].astype(float) >= keep_thr].itertuples(index=False):\n                # Boundary-safe evidence should include either actual point\n                # alignment or a stacking gain; pure contour correlation is not enough.\n                if float(row.point_score) < 0.18 and float(row.stack_gain) < 0.42:\n                    continue\n                uf.union(int(row.image_id_a), int(row.image_id_b))\n\n    if merge_rank > 0:\n        cross = pair_scores[~pair_scores[\"same_current_cluster\"].astype(bool)].copy()\n        if not cross.empty:\n            cross = cross[\n                (cross[\"score\"].astype(float) >= merge_thr)\n                & (cross[\"overlap\"].astype(float) >= 0.105)\n                & (cross[\"point_score\"].astype(float) >= 0.42)\n                & (cross[\"stack_gain\"].astype(float) >= 0.46)\n            ].copy()\n            if not cross.empty:\n                neighbors: dict[int, list[tuple[int, float]]] = {i: [] for i in ids}\n                for row in cross.itertuples(index=False):\n                    a = int(row.image_id_a)\n                    b = int(row.image_id_b)\n                    s = float(row.score)\n                    neighbors[a].append((b, s))\n                    neighbors[b].append((a, s))\n                ranks: dict[tuple[int, int], int] = {}\n                for node, vals in neighbors.items():\n                    vals.sort(key=lambda x: -x[1])\n                    for rank, (other, _) in enumerate(vals, start=1):\n                        ranks[(node, other)] = rank\n                for row in cross.sort_values(\"score\", ascending=False).itertuples(index=False):\n                    a = int(row.image_id_a)\n                    b = int(row.image_id_b)\n                    if current_by_id[a] == current_by_id[b]:\n                        continue\n                    if ranks.get((a, b), 999) <= merge_rank and ranks.get((b, a), 999) <= merge_rank:\n                        ra = uf.find(a)\n                        rb = uf.find(b)\n                        if ra != rb and uf.size[ra] + uf.size[rb] <= max_cross_component:\n                            uf.union(ra, rb)\n\n    components: dict[int, list[int]] = {}\n    for image_id in ids:\n        components.setdefault(uf.find(image_id), []).append(image_id)\n    current_sets = {cluster: set(members) for cluster, members in by_cluster.items()}\n    comp_order: dict[int, int] = {}\n    labels: dict[int, str] = {}\n    for root, members in sorted(components.items(), key=lambda kv: min(kv[1])):\n        member_set = set(members)\n        member_current = {current_by_id[i] for i in members}\n        if len(member_current) == 1:\n            current_cluster = next(iter(member_current))\n            if member_set == current_sets.get(current_cluster, set()):\n                for image_id in members:\n                    labels[image_id] = current_cluster\n                continue\n        comp_order[root] = len(comp_order)\n        new_label = f\"cluster_TexasHornedLizards_boundary_{variant}_{comp_order[root]}\"\n        for image_id in members:\n            labels[image_id] = new_label\n    return labels\n\n\ndef main() -> None:\n    base.VERSION = VERSION\n    base.find_current_best = find_current_best_boundary\n    base.align_vertical = align_vertical_no_flip\n    base.texas_belly_template = texas_belly_template_boundary\n    base.texas_pair_score = texas_pair_score_boundary\n    base.texas_variant_labels = texas_variant_labels_boundary\n    base.main()\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
Path("texas_boundary_splitonly_final_v20260430.py").write_text("from __future__ import annotations\n\nimport argparse\nimport itertools\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nimport texas_astrodot_2025reuse_v20260426 as base\nimport texas_astrodot_boundary_suppressed_v20260429 as boundary\n\n\nVERSION = \"texas_boundary_splitonly_final_v20260430\"\nTEXAS = base.TEXAS\nSPECIES_ORDER = base.core.SPECIES_ORDER\n\n\nclass UnionFind:\n    def __init__(self, values):\n        self.parent = {int(v): int(v) for v in values}\n        self.size = {int(v): 1 for v in values}\n\n    def find(self, value: int) -> int:\n        value = int(value)\n        root = value\n        while self.parent[root] != root:\n            root = self.parent[root]\n        while self.parent[value] != value:\n            nxt = self.parent[value]\n            self.parent[value] = root\n            value = nxt\n        return root\n\n    def union(self, a: int, b: int) -> None:\n        ra = self.find(a)\n        rb = self.find(b)\n        if ra == rb:\n            return\n        if self.size[ra] < self.size[rb]:\n            ra, rb = rb, ra\n        self.parent[rb] = ra\n        self.size[ra] += self.size[rb]\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description=(\n            \"Final Texas split-only verifier. Starts from the current 0.32583 \"\n            \"submission, computes boundary-suppressed Texas belly-dot scores, \"\n            \"and only splits existing Texas clusters. No cross-cluster merges.\"\n        )\n    )\n    parser.add_argument(\"--data-root\", type=str, default=None)\n    parser.add_argument(\"--sam-manifest\", type=str, default=None)\n    parser.add_argument(\"--current-best-submission\", type=str, default=None)\n    parser.add_argument(\"--output-root\", type=str, default=None)\n    parser.add_argument(\"--max-side\", type=int, default=760)\n    parser.add_argument(\"--texas-canvas-w\", type=int, default=224)\n    parser.add_argument(\"--texas-canvas-h\", type=int, default=320)\n    parser.add_argument(\"--save-visualizations\", action=\"store_true\")\n    parser.add_argument(\"--visual-limit\", type=int, default=18)\n    parser.add_argument(\"--smoke\", action=\"store_true\")\n    return parser.parse_args()\n\n\ndef load_submission(path: Path) -> pd.DataFrame:\n    df = pd.read_csv(path)\n    if \"image_id\" not in df.columns or \"cluster\" not in df.columns:\n        raise ValueError(f\"{path} must contain image_id and cluster columns\")\n    out = df[[\"image_id\", \"cluster\"]].copy()\n    out[\"image_id\"] = out[\"image_id\"].astype(int)\n    out[\"cluster\"] = out[\"cluster\"].astype(str)\n    return out\n\n\ndef score_within_current_clusters(items: list[base.TexasDotItem]) -> pd.DataFrame:\n    by_cluster: dict[str, list[base.TexasDotItem]] = {}\n    for item in items:\n        by_cluster.setdefault(item.current_cluster, []).append(item)\n    rows = []\n    total = sum(len(v) * (len(v) - 1) // 2 for v in by_cluster.values())\n    done = 0\n    for cluster, members in sorted(by_cluster.items(), key=lambda kv: (min(it.image_id for it in kv[1]), kv[0])):\n        if len(members) < 2:\n            continue\n        for a, b in itertools.combinations(sorted(members, key=lambda it: it.image_id), 2):\n            score = boundary.texas_pair_score_boundary(a, b)\n            rows.append(\n                {\n                    \"species\": TEXAS,\n                    \"image_id_a\": int(a.image_id),\n                    \"image_id_b\": int(b.image_id),\n                    \"current_cluster\": cluster,\n                    \"current_cluster_a\": a.current_cluster,\n                    \"current_cluster_b\": b.current_cluster,\n                    **score,\n                }\n            )\n            done += 1\n            if done % 1000 == 0:\n                print(f\"[Texas split-only] scored {done}/{total} intra-cluster pairs\")\n    return pd.DataFrame(rows)\n\n\ndef preserve_cluster(members: list[int], label: str) -> dict[int, str]:\n    return {int(i): label for i in members}\n\n\ndef split_cluster_by_edges(\n    members: list[int],\n    old_label: str,\n    pair_scores: pd.DataFrame,\n    variant: str,\n    keep_thr: float,\n    min_point: float,\n    min_stack: float,\n    preserve_up_to: int,\n    min_edges_per_component: int,\n) -> tuple[dict[int, str], dict]:\n    members = sorted(map(int, members))\n    n = len(members)\n    if n <= preserve_up_to:\n        return preserve_cluster(members, old_label), {\n            \"variant\": variant,\n            \"old_cluster\": old_label,\n            \"old_size\": n,\n            \"action\": \"preserve_small\",\n            \"new_sizes\": str(n),\n            \"accepted_edges\": 0,\n        }\n\n    inside = pair_scores[pair_scores[\"current_cluster\"].eq(old_label)].copy()\n    if inside.empty:\n        return preserve_cluster(members, old_label), {\n            \"variant\": variant,\n            \"old_cluster\": old_label,\n            \"old_size\": n,\n            \"action\": \"preserve_no_scores\",\n            \"new_sizes\": str(n),\n            \"accepted_edges\": 0,\n        }\n\n    # Keep only interior belly-dot evidence. Correlation alone is not enough:\n    # the whole point is avoiding silhouette-driven false links.\n    accepted = inside[\n        (inside[\"score\"].astype(float) >= keep_thr)\n        & (\n            (inside[\"point_score\"].astype(float) >= min_point)\n            | (inside[\"stack_gain\"].astype(float) >= min_stack)\n        )\n    ].copy()\n\n    if accepted.empty:\n        return preserve_cluster(members, old_label), {\n            \"variant\": variant,\n            \"old_cluster\": old_label,\n            \"old_size\": n,\n            \"action\": \"preserve_no_trusted_edges\",\n            \"new_sizes\": str(n),\n            \"accepted_edges\": 0,\n        }\n\n    uf = UnionFind(members)\n    for row in accepted.sort_values([\"score\", \"point_score\", \"stack_gain\"], ascending=False).itertuples(index=False):\n        uf.union(int(row.image_id_a), int(row.image_id_b))\n\n    comps: dict[int, list[int]] = {}\n    for image_id in members:\n        comps.setdefault(uf.find(image_id), []).append(image_id)\n    parts = sorted([sorted(v) for v in comps.values()], key=lambda xs: (min(xs), len(xs), max(xs)))\n\n    if len(parts) == 1:\n        return preserve_cluster(members, old_label), {\n            \"variant\": variant,\n            \"old_cluster\": old_label,\n            \"old_size\": n,\n            \"action\": \"preserve_connected\",\n            \"new_sizes\": str(n),\n            \"accepted_edges\": int(len(accepted)),\n        }\n\n    # Avoid destructive atomization. If most parts are isolated and the cluster\n    # is not huge, this may just be poor image quality rather than a false merge.\n    singleton_parts = sum(1 for p in parts if len(p) == 1)\n    if singleton_parts >= len(parts) - 1 and n <= 7:\n        return preserve_cluster(members, old_label), {\n            \"variant\": variant,\n            \"old_cluster\": old_label,\n            \"old_size\": n,\n            \"action\": \"preserve_atomization_guard\",\n            \"new_sizes\": \" \".join(map(str, [len(p) for p in parts])),\n            \"accepted_edges\": int(len(accepted)),\n        }\n\n    # Require at least a tiny amount of support inside every multi-image part.\n    # Singletons are allowed because they are the suspected weak outliers.\n    if min_edges_per_component > 0:\n        for part in parts:\n            if len(part) <= 1:\n                continue\n            part_set = set(part)\n            edge_count = int(\n                accepted[\"image_id_a\"].isin(part_set).mul(accepted[\"image_id_b\"].isin(part_set)).sum()\n            )\n            if edge_count < min_edges_per_component:\n                return preserve_cluster(members, old_label), {\n                    \"variant\": variant,\n                    \"old_cluster\": old_label,\n                    \"old_size\": n,\n                    \"action\": \"preserve_component_support_guard\",\n                    \"new_sizes\": \" \".join(map(str, [len(p) for p in parts])),\n                    \"accepted_edges\": int(len(accepted)),\n                }\n\n    out: dict[int, str] = {}\n    for idx, part in enumerate(parts):\n        new_label = f\"{old_label}__{variant}_{idx}\"\n        for image_id in part:\n            out[image_id] = new_label\n    return out, {\n        \"variant\": variant,\n        \"old_cluster\": old_label,\n        \"old_size\": n,\n        \"action\": \"split\",\n        \"new_sizes\": \" \".join(map(str, [len(p) for p in parts])),\n        \"accepted_edges\": int(len(accepted)),\n    }\n\n\ndef splitonly_labels(\n    items: list[base.TexasDotItem],\n    pair_scores: pd.DataFrame,\n    variant: str,\n    keep_thr: float,\n    min_point: float,\n    min_stack: float,\n    preserve_up_to: int,\n    min_edges_per_component: int,\n) -> tuple[dict[int, str], pd.DataFrame]:\n    by_cluster: dict[str, list[int]] = {}\n    for item in items:\n        by_cluster.setdefault(item.current_cluster, []).append(int(item.image_id))\n    labels: dict[int, str] = {}\n    reports = []\n    for old_label, members in sorted(by_cluster.items(), key=lambda kv: (min(kv[1]), len(kv[1]), kv[0])):\n        cluster_labels, report = split_cluster_by_edges(\n            members,\n            old_label,\n            pair_scores,\n            variant,\n            keep_thr,\n            min_point,\n            min_stack,\n            preserve_up_to,\n            min_edges_per_component,\n        )\n        labels.update(cluster_labels)\n        reports.append(report)\n    return labels, pd.DataFrame(reports)\n\n\ndef compact_labels(submission: pd.DataFrame, test_rows: pd.DataFrame) -> pd.DataFrame:\n    species_map = dict(zip(test_rows[\"image_id\"].astype(int), test_rows[\"species_id\"].astype(str)))\n    labels = dict(zip(submission[\"image_id\"].astype(int), submission[\"cluster\"].astype(str)))\n    out_map: dict[int, str] = {}\n    for species in SPECIES_ORDER:\n        ids = sorted([i for i, sp in species_map.items() if sp == species])\n        groups: dict[str, list[int]] = {}\n        for image_id in ids:\n            groups.setdefault(labels[image_id], []).append(image_id)\n        for idx, members in enumerate(sorted(groups.values(), key=lambda xs: (min(xs), len(xs), max(xs)))):\n            label = f\"cluster_{species}_{idx}\"\n            for image_id in members:\n                out_map[image_id] = label\n    out = submission.copy()\n    out[\"cluster\"] = out[\"image_id\"].astype(int).map(out_map)\n    if out[\"cluster\"].isna().any():\n        raise ValueError(\"Compacted labels contain missing values\")\n    return out\n\n\ndef build_submission(\n    current: pd.DataFrame,\n    test_rows: pd.DataFrame,\n    texas_labels: dict[int, str],\n    out_path: Path,\n) -> pd.DataFrame:\n    current_map = dict(zip(current[\"image_id\"].astype(int), current[\"cluster\"].astype(str)))\n    texas_ids = set(test_rows.loc[test_rows[\"species_id\"].eq(TEXAS), \"image_id\"].astype(int).tolist())\n    sub = current.copy()\n    sub[\"cluster\"] = sub[\"image_id\"].astype(int).map(\n        lambda i: texas_labels.get(i, current_map[i]) if i in texas_ids else current_map[i]\n    )\n    sub = compact_labels(sub, test_rows)\n    sub.to_csv(out_path, index=False)\n    return sub\n\n\ndef summarize_submission(sub: pd.DataFrame, current: pd.DataFrame, test_rows: pd.DataFrame, variant: str) -> list[dict]:\n    cur_map = dict(zip(current[\"image_id\"].astype(int), current[\"cluster\"].astype(str)))\n    sub_map = dict(zip(sub[\"image_id\"].astype(int), sub[\"cluster\"].astype(str)))\n    rows = []\n    for species in SPECIES_ORDER:\n        ids = test_rows.loc[test_rows[\"species_id\"].eq(species), \"image_id\"].astype(int).tolist()\n        labels = [sub_map[i] for i in ids]\n        counts = pd.Series(labels).value_counts()\n        cur_groups: dict[str, set[int]] = {}\n        sub_groups: dict[str, set[int]] = {}\n        for image_id in ids:\n            cur_groups.setdefault(cur_map[image_id], set()).add(image_id)\n            sub_groups.setdefault(sub_map[image_id], set()).add(image_id)\n        cur_membership = {image_id: cur_groups[cur_map[image_id]] for image_id in ids}\n        sub_membership = {image_id: sub_groups[sub_map[image_id]] for image_id in ids}\n        partition_changed = int(sum(1 for image_id in ids if cur_membership[image_id] != sub_membership[image_id]))\n        rows.append(\n            {\n                \"variant\": variant,\n                \"species\": species,\n                \"n_images\": int(len(ids)),\n                \"n_clusters\": int(counts.shape[0]),\n                \"singletons\": int((counts == 1).sum()),\n                \"max_cluster_size\": int(counts.max()) if not counts.empty else 0,\n                \"rows_changed_vs_current\": partition_changed,\n            }\n        )\n    return rows\n\n\ndef validate_upload(submission: pd.DataFrame, sample: pd.DataFrame) -> None:\n    if list(submission.columns) != [\"image_id\", \"cluster\"]:\n        raise ValueError(\"Submission columns must be exactly image_id, cluster\")\n    if submission[\"image_id\"].astype(int).tolist() != sample[\"image_id\"].astype(int).tolist():\n        raise ValueError(\"Submission image order does not match sample_submission.csv\")\n    if submission[\"cluster\"].isna().any():\n        raise ValueError(\"Submission contains missing cluster labels\")\n    max_len = int(submission[\"cluster\"].astype(str).str.len().max())\n    if max_len > 64:\n        raise ValueError(f\"Cluster labels too long: max length {max_len}\")\n\n\ndef choose_safe_candidate(candidate_report: pd.DataFrame) -> str:\n    texas = candidate_report[candidate_report[\"species\"].eq(TEXAS)].copy()\n    safe = texas[\n        (texas[\"rows_changed_vs_current\"].astype(int) > 0)\n        & (texas[\"rows_changed_vs_current\"].astype(int) <= 70)\n        & (texas[\"n_clusters\"].astype(int).between(80, 92))\n        & (texas[\"max_cluster_size\"].astype(int) <= 26)\n    ].copy()\n    priority = {\"balanced\": 0, \"strict\": 1, \"mild\": 2}\n    if safe.empty:\n        return \"current_passthrough\"\n    safe[\"priority\"] = safe[\"variant\"].map(priority).fillna(99).astype(int)\n    safe = safe.sort_values([\"priority\", \"rows_changed_vs_current\", \"n_clusters\"])\n    return str(safe.iloc[0][\"variant\"])\n\n\ndef main() -> None:\n    args = parse_args()\n    if args.smoke:\n        args.save_visualizations = True\n\n    data_root = base.find_data_root(args.data_root)\n    sam_manifest = base.find_sam_manifest(args.sam_manifest)\n    metadata, manifest_info = base.prepare_metadata(data_root, sam_manifest)\n    metadata = metadata[metadata[\"species_id\"].isin(SPECIES_ORDER)].copy()\n    test_rows = metadata[metadata[\"split\"].eq(\"test\")].copy()\n    sample = pd.read_csv(data_root / \"sample_submission.csv\")[[\"image_id\"]].copy()\n    sample[\"image_id\"] = sample[\"image_id\"].astype(int)\n\n    output_root = Path(args.output_root) if args.output_root else Path.cwd() / f\"animalclef_{VERSION}\"\n    reports_dir = output_root / \"reports\"\n    sub_dir = output_root / \"submissions\"\n    viz_dir = output_root / \"visualizations\"\n    reports_dir.mkdir(parents=True, exist_ok=True)\n    sub_dir.mkdir(parents=True, exist_ok=True)\n    if args.save_visualizations:\n        viz_dir.mkdir(parents=True, exist_ok=True)\n\n    current_best_path = boundary.find_current_best_boundary(args.current_best_submission, data_root)\n    current = load_submission(current_best_path)\n    validate_upload(current, sample)\n    current_labels = dict(zip(current[\"image_id\"].astype(int), current[\"cluster\"].astype(str)))\n\n    print(f\"VERSION={VERSION}\")\n    print(f\"data_root={data_root}\")\n    print(f\"sam_manifest={sam_manifest}\")\n    print(f\"current_best={current_best_path}\")\n    print(f\"output_root={output_root}\")\n    print(json.dumps(manifest_info, indent=2))\n\n    texas_rows = test_rows[test_rows[\"species_id\"].eq(TEXAS)].sort_values(\"image_id\").copy()\n    if args.smoke:\n        texas_rows = texas_rows.head(42)\n\n    items: list[base.TexasDotItem] = []\n    for idx, row in enumerate(texas_rows.to_dict(\"records\"), start=1):\n        image_id = int(row[\"image_id\"])\n        try:\n            items.append(boundary.texas_belly_template_boundary(row, current_labels[image_id], args))\n        except Exception as exc:\n            print(f\"[warn] Texas template failed image_id={image_id}: {exc}\")\n        if idx % 75 == 0:\n            print(f\"[Texas split-only] templates {idx}/{len(texas_rows)}\")\n\n    if args.save_visualizations:\n        base.save_texas_preview(items, viz_dir / f\"{VERSION}_Texas_template_preview.jpg\", args.visual_limit)\n\n    pair_scores = score_within_current_clusters(items)\n    pair_scores.to_csv(reports_dir / f\"{VERSION}_Texas_intra_cluster_pair_scores.csv\", index=False)\n\n    variants = {\n        \"mild\": dict(keep_thr=0.285, min_point=0.18, min_stack=0.38, preserve_up_to=6, min_edges_per_component=0),\n        \"balanced\": dict(keep_thr=0.310, min_point=0.24, min_stack=0.42, preserve_up_to=4, min_edges_per_component=0),\n        \"strict\": dict(keep_thr=0.340, min_point=0.30, min_stack=0.46, preserve_up_to=3, min_edges_per_component=0),\n    }\n\n    candidate_rows: list[dict] = []\n    split_reports = []\n    outputs: dict[str, str] = {}\n\n    # Keep a no-op copy so the notebook can fall back safely if shape guards fail.\n    passthrough_path = sub_dir / f\"submission_{VERSION}_current_passthrough.csv\"\n    current_compact = compact_labels(current, test_rows)\n    current_compact.to_csv(passthrough_path, index=False)\n    outputs[\"current_passthrough\"] = str(passthrough_path)\n    candidate_rows.extend(summarize_submission(current_compact, current, test_rows, \"current_passthrough\"))\n\n    for variant, params in variants.items():\n        labels, split_report = splitonly_labels(items, pair_scores, variant=variant, **params)\n        split_report[\"variant\"] = variant\n        split_reports.append(split_report)\n        out_path = sub_dir / f\"submission_{VERSION}_{variant}.csv\"\n        sub = build_submission(current, test_rows, labels, out_path)\n        validate_upload(sub, sample)\n        outputs[variant] = str(out_path)\n        candidate_rows.extend(summarize_submission(sub, current, test_rows, variant))\n        print(f\"wrote {out_path}\")\n\n    candidate_report = pd.DataFrame(candidate_rows)\n    candidate_report.to_csv(reports_dir / f\"{VERSION}_candidate_report.csv\", index=False)\n    if split_reports:\n        pd.concat(split_reports, ignore_index=True).to_csv(reports_dir / f\"{VERSION}_split_report.csv\", index=False)\n    else:\n        pd.DataFrame().to_csv(reports_dir / f\"{VERSION}_split_report.csv\", index=False)\n\n    selected = choose_safe_candidate(candidate_report)\n    selected_path = Path(outputs[selected])\n    final = pd.read_csv(selected_path)\n    validate_upload(final, sample)\n    top_level = output_root / \"submission.csv\"\n    final.to_csv(top_level, index=False)\n\n    selection = {\n        \"selected_variant\": selected,\n        \"selected_path\": str(selected_path),\n        \"top_level_submission\": str(top_level),\n        \"outputs\": outputs,\n        \"manifest_info\": manifest_info,\n    }\n    (reports_dir / f\"{VERSION}_selected_submission.json\").write_text(json.dumps(selection, indent=2), encoding=\"utf-8\")\n\n    summary = {\n        \"version\": VERSION,\n        \"data_root\": str(data_root),\n        \"sam_manifest\": str(sam_manifest) if sam_manifest else None,\n        \"current_best\": str(current_best_path),\n        \"manifest_info\": manifest_info,\n        \"texas_items\": int(len(items)),\n        \"texas_intra_pairs\": int(len(pair_scores)),\n        \"selected_variant\": selected,\n        \"outputs\": outputs,\n    }\n    (reports_dir / f\"{VERSION}_summary.json\").write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n\n    print(\"\\nCandidate report:\")\n    print(candidate_report.to_string(index=False))\n    print(\"\\nSelected variant:\", selected)\n    print(\"Top-level submission:\", top_level)\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
print("embedded scripts written")
            

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("cv2") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=True)
else:
    print("OpenCV already available")
            

In [ ]:
from pathlib import Path
import subprocess
import sys

OUTPUT_ROOT = Path("/kaggle/working/animalclef_texas_boundary_splitonly_final_v20260430")

cmd = [
    sys.executable,
    "texas_boundary_splitonly_final_v20260430.py",
    "--output-root", str(OUTPUT_ROOT),
    "--save-visualizations",
]

print("running:", " ".join(cmd))
subprocess.run(cmd, check=True)
            

In [ ]:
from pathlib import Path
import json
import pandas as pd

out = Path("/kaggle/working/animalclef_texas_boundary_splitonly_final_v20260430")
reports = out / "reports"
subs = out / "submissions"

candidate_report = pd.read_csv(reports / "texas_boundary_splitonly_final_v20260430_candidate_report.csv")
split_report = pd.read_csv(reports / "texas_boundary_splitonly_final_v20260430_split_report.csv")
selection = json.loads((reports / "texas_boundary_splitonly_final_v20260430_selected_submission.json").read_text())

display(candidate_report)
display(split_report[split_report["action"].eq("split")])
print(json.dumps(selection, indent=2))

submit_path = out / "submission.csv"
df = pd.read_csv(submit_path)
print("Submit candidate:", submit_path)
print("Selected variant:", selection["selected_variant"])
print("rows:", len(df))
print("max label length:", df["cluster"].astype(str).str.len().max())
display(df.head())
            

In [ ]:
from pathlib import Path
from IPython.display import Image as IPImage, display

viz = Path("/kaggle/working/animalclef_texas_boundary_splitonly_final_v20260430/visualizations")
for path in sorted(viz.glob("*.jpg")):
    print(path.name)
    display(IPImage(filename=str(path)))
            